# TexHype V7 — Multi-Split Evaluation with Random Forest

This Notebook contains a ML model estimating pain and dehydration.
It is structured as follows:
1. General Data Processing
2. Pain Estimation based on Smartwatch data
3. Dehydration Estimation based on Smartwatch data



### 1. General Data Processing

In [ ]:
#### Import Modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import os
import random
import shap
import xgboost as xgb
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import roc_curve, auc
from sklearn.metrics import precision_recall_curve, average_precision_score, f1_score, brier_score_loss


In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

##### Load Data

#### 1.1 Smartwatch Data

In [ ]:
#load pre-processed data
all_data = pd.read_csv('output_all.csv')
personal_data = pd.read_csv('personal_data.csv')

##### Overview of Data

In [ ]:
#Calculate Age and Gender in personal_data
year_of_measurement = 2024
age =  year_of_measurement - personal_data['Year of Birth']
personal_data['Age'] = age

#Map Gender: female:0 male: 1
personal_data['Gender'] = personal_data['Sex'].map({'HKBiologicalSexFemale': 0, 'HKBiologicalSexMale':1})

##### Pivot Table

In [ ]:
# flip rows and columns in all_data
all_data = all_data.pivot_table(
                       index = ['ID', 'Start', 'End', 'enrollment_date', 'final_visit'],
                       columns = 'Name',
                       values = 'Value',
                       aggfunc = 'mean',
                      ).reset_index()

#merge with personal data
df = all_data.merge(personal_data[['ID', 'Gender', 'Age']], on = 'ID', how = 'left')

##### Rename Columns

In [ ]:
df.columns = df.columns.str.replace('HKCategoryTypeIdentifier', "", regex=True)
df.columns = df.columns.str.replace('HKQuantityTypeIdentifier', "", regex=True)

##### Fill Constants

In [ ]:
# Fill Weight and Height for every row
df[['Height','BodyMass']] = df.groupby('ID')[['Height','BodyMass']].ffill().bfill()

#### Cut Data

In [ ]:
def time_align(df):
   
    date_cols = ['Start', 'End', 'enrollment_date', 'final_visit']
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')

    #only keep data between erollment and final visit date
    mask = ((df['Start'].dt.date >= df['enrollment_date'].dt.date) &(df['End'].dt.date <= df['final_visit'].dt.date))
    df = df.loc[mask].copy()

    df = df.drop(columns=['enrollment_date', 'final_visit']).reset_index(drop=True)

    return df
df = time_align(df)

#### Time Window

In [ ]:
#Aim: Only use data measured on days where parameters used for the label defintion were measured as well
duration = 120 #min

def time_window(df, duration):
    df['Start'] = pd.to_datetime(df['Start'], errors='coerce')
    df = df.sort_values(['ID', 'Start']).copy()

    #define time window
    tw = pd.Timedelta(minutes = duration) #minutes per time window
    hr = df[df['HeartRate'].notna()] #take Heart rate as guidance for time window generation. Take all HR values not NaN

    #start of time window is first Heart rate measurement and end is last measurement
    first_ = hr.groupby('ID')['Start'].min()
    last_ = hr.groupby('ID')['Start'].max()

    #time windows per ID
    tw_list = []
    for id_ in df['ID'].unique():
        if id_ not in first_.index:
            print(f"Warning: ID {id_} has no HeartRate data, skipping.")
            continue
        start_time = first_[id_]
        end_time = last_[id_]
        n_windows = int(((end_time-start_time).total_seconds())//(duration*60))+1
        #generate all tws per ID
        for tw_num in range(n_windows):
            window_start = start_time + pd.Timedelta(minutes = tw_num*duration)
            window_end = window_start+tw
            tw_list.append((id_, tw_num, window_start, window_end))
    #make df
    tw_df = pd.DataFrame(tw_list, columns=['ID','time_window','window_start','window_end'])

    #start of time windows is first Heart rate measurement
    df['_start']= df['ID'].map(first_)
    df['_end'] = df['ID'].map(last_)

    #only take values after first HR measurement
    mask = (df['Start']>=df['_start'])&(df['Start']<=df['_end'])
    df = df.loc[mask].copy()

    #calculate time window
    delta = (df['Start']-df['_start']).dt.total_seconds()
    df['time_window']=(delta//(duration*60)).astype('Int64')

    #drop _start and _end
    df.drop(columns=['_start', '_end'], inplace=True)

    return df, tw_df
    
df, tw_df = time_window(df, duration)

#### Aggregate

In [ ]:
def agg_values(df, tw_df):
    group = df.groupby(['ID', 'time_window'])

    df = group.agg({
        'Age':'first',
        'Gender': 'first',
        'Height': 'first',
        'BodyMass': 'first',
        'HeartRate': ['max', 'median', 'min', 'std'],
        'OxygenSaturation': ['max', 'median', 'min'],
        'HeartRateVariabilitySDNN': ['max', 'median', 'min'],
        'RestingHeartRate': ['median'],
        'ActiveEnergyBurned': ['max', 'median', 'min', 'sum'],
        'BasalEnergyBurned': ['max', 'median', 'min', 'sum'],
        'PhysicalEffort': ['max', 'median', 'min', 'sum'],
        'AppleStandHour': ['sum'], 
        'AppleStandTime':['sum'],
        'AppleExerciseTime': ['sum'],
        'DistanceWalkingRunning': ['sum'],
        'StepCount': ['sum'],
        'WalkingStepLength': ['median'],
        'WalkingSpeed': ['median'],
        'WalkingAsymmetryPercentage': ['median'],
        'WalkingDoubleSupportPercentage': ['median'],
        'WalkingHeartRateAverage': ['median'],
        'FlightsClimbed': ['sum'],
        'StairAscentSpeed': ['median'],
        'StairDescentSpeed':['median'],
        'SixMinuteWalkTestDistance': ['median']
    })

    df = df.reset_index()

    #Rename multi_index columns
    df.columns = [
        col[0] if isinstance(col, tuple) and col[1] == '' else f"{col[0]}_{col[1]}"
        for col in df.columns
    ]
    #Combine all data and time window dataframe to have all time windows, even those without values
    df = pd.merge(tw_df, df, on=['ID', 'time_window'], how = 'left')

    return df

#aggregate all values within each time window
df = agg_values(df, tw_df)

#### Cut inside Data

In [ ]:
#define search window to cut out times where smartwatch was not worn
sw = 1 # length of search window in multiples of duration. 4*15min = 1h

def cut_inside(df, sw): #
    df['Needed'] = 1 # for 1 = needed, 0= not needed
    save_params = ['ID', 'Age_first', 'Gender_first', 'Height_first', 'BodyMass_first', 'time_window', 'window_start', 'window_end']
    cols_to_nan = [col for col in df.columns if col not in save_params]
    result =[]

    groups = df.groupby('ID')

    for id_, group in groups:
        start = group['time_window'].min()
        finish = group['time_window'].max()

        while start <= finish:
            sw_end = start + sw

            mask = (group['time_window']>=start) & (group['time_window']< sw_end)
            searched_vals = group[mask]
            hr = searched_vals['HeartRate_median']

            #majority vote
            if hr.isna().sum() >len(hr)/2:
                #set all params to NaN and mark them as not needed
                group.loc[mask, cols_to_nan] = pd.NA
                group.loc[mask, 'Needed'] = 0
            else:
                group.loc[mask, 'Needed'] = 1
            start = sw_end

        result.append(group)
    df = pd.concat(result).sort_index()

    return df

df = cut_inside(df, sw)

#### Impute

In [ ]:
def impute_vals(df):
    #For truly count/cumulative event-based measurements, impute zero if NaN
    eventbased_cols = ['AppleStandHour_sum', 'AppleStandTime_sum', 'AppleExerciseTime_sum', 'DistanceWalkingRunning_sum',
                       'StepCount_sum',
                       'FlightsClimbed_sum']
    df[eventbased_cols] = df[eventbased_cols].fillna(0)

    #For median-based gait/walk metrics, forward fill then per-ID mean (NOT zero — 0 is physiologically impossible)
    gait_cols = ['WalkingStepLength_median',
                 'WalkingSpeed_median', 'WalkingAsymmetryPercentage_median',
                 'WalkingDoubleSupportPercentage_median',
                 'WalkingHeartRateAverage_median',
                 'StairAscentSpeed_median', 'StairDescentSpeed_median',
                 'SixMinuteWalkTestDistance_median']
    df[gait_cols] = df.groupby('ID')[gait_cols].ffill(limit=3)
    df[gait_cols] = df.groupby('ID')[gait_cols].transform(lambda x: x.fillna(x.mean()))
    df[gait_cols] = df[gait_cols].fillna(0)  # last resort: patient never walked during study

    #for all other non-eventbased measurements do forward fill (limited to 3 windows max)
    noteventbased_cols = ['HeartRate_max', 'HeartRate_median', 'HeartRate_min', 'HeartRate_std',
                          'OxygenSaturation_max', 'OxygenSaturation_median', 'OxygenSaturation_min',
                          'HeartRateVariabilitySDNN_max', 'HeartRateVariabilitySDNN_median', 
                          'HeartRateVariabilitySDNN_min', 'RestingHeartRate_median', 
                          'ActiveEnergyBurned_max', 'ActiveEnergyBurned_median',
                          'ActiveEnergyBurned_min', 'ActiveEnergyBurned_sum', 'BasalEnergyBurned_max',
                          'BasalEnergyBurned_median', 'BasalEnergyBurned_min',
                          'BasalEnergyBurned_sum', 'PhysicalEffort_max', 'PhysicalEffort_median', 'PhysicalEffort_min',
                          'PhysicalEffort_sum']
    df[noteventbased_cols] = df.groupby('ID')[noteventbased_cols].ffill(limit=3)
    #for NaN fill columns with mean of column per ID
    df[noteventbased_cols] = df.groupby('ID')[noteventbased_cols].transform(
        lambda x: x.fillna(x.mean())
    )
    # for e.g. ID10 mean of resting heartrate is NaN - impute with global mean (not per ID)
    df[noteventbased_cols] = df[noteventbased_cols].fillna(df[noteventbased_cols].mean())

    #for Age, Gender, Weight and height do forward and backward fill since they are constant
    constant_cols = ['Age_first', 'Gender_first', 'Height_first', 'BodyMass_first']
    df[constant_cols] = (df.groupby('ID')[constant_cols].ffill().bfill())

    #Add difference
    for ele in ['HeartRate_median', 'OxygenSaturation_median']:
        df[f'diff_{ele}'] = df.groupby(['ID'])[ele].diff().fillna(0)

    noteventbased_cols = noteventbased_cols + ['diff_HeartRate_median', 'diff_OxygenSaturation_median']

    return df, noteventbased_cols

#impute values
df, noteventbased_cols = impute_vals(df)

#### Clean Data before Labels

In [ ]:
def clean_data(df):
    # drop all rows where needed = 0
    df = df[df['Needed']!=0].reset_index(drop=True)

    # drop information about time_window, start, end
    drop_cols = ['Needed'] #'time_window', 'window_start', 'window_end',
    df = df.drop(columns = drop_cols)
    
    return df

df = clean_data(df)

#### Add Lags

In [ ]:
# jetzt schauen wie es für sum_cols klappt
def add_lags(df, noteventbased_cols):
    lags = [1, 3, 6] # time window number #change them for bigger lags
    diff_cols = ['HeartRate_median','OxygenSaturation_median','HeartRateVariabilitySDNN_median','RestingHeartRate_median']
    sum_cols = ['ActiveEnergyBurned_sum', 'BasalEnergyBurned_sum','PhysicalEffort_sum']

    df = df.copy()
    df = df.sort_values(by=['ID', 'time_window'])

    for lag in lags:
        #diff_cols
        shifted = df.groupby('ID')[diff_cols].shift(lag)
        

        #diff (value(t)-value(t_lag))
        diff = df[diff_cols]-shifted  
        diff.columns = [f'{col}_lag{lag}' for col in diff_cols]

        df = pd.concat([df, diff], axis=1)
    #for sum cols.
    #sum_cols — use groupby rolling to avoid crossing patient boundaries
    shifted = df.groupby('ID')[sum_cols].shift(1)
    rolling_sum = shifted.groupby(df['ID']).rolling(window=lags[2], min_periods=1).sum().droplevel(0)
    rolling_sum.columns=[f'{col}_sum_lag' for col in sum_cols]
    df = pd.concat([df, rolling_sum], axis=1)
    
    #impute lags that are not computable
    #the values before the first measurement can not be attached as lags to the df. The added lines are NaN.
    # They get forward filled and if there are still NaNs left, the mean of the column is imputed
    lag_cols = ([f'{col}_lag{lag}' for lag in lags for col in diff_cols]+[f'{col}_sum_lag' for col in sum_cols])
    
    #group by ID, impute mean for NaN per ID
    for col in lag_cols:
       df[col] = df.groupby('ID')[col].transform(
           lambda x: x.ffill().fillna(x.mean())
       )
        
       #ID10 only has 4 datapoints - impute NaN with global mean
       df[col] = df[col].fillna(df[col].mean())
    
    
    

    return df

df = add_lags(df, noteventbased_cols)

In [ ]:
df.isna().sum()

#### 1.2. Label Data

#### Load Label Data

In [ ]:
df_label = pd.read_csv('export.csv')

#### Check Data

In [ ]:
df_label.describe()
df_label.boxplot(by='study_id', column=['vas_score']) #vena_cava_diameter_mm

In [ ]:
df_label.boxplot(by='study_id', column=['vena_cava_diameter_mm']) #vena_cava_diameter_mm

#### Outliers

In [ ]:
def detect_outlier(df_label):
    groups = df_label.groupby('study_id')
    out_list=[]

    for key, group in groups:
        numeric_cols = group.select_dtypes(include=np.number).columns
        for col in numeric_cols:
            q1 = group[col].quantile(0.25)
            q3 = group[col].quantile(0.75)
            iqr = q3 - q1
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            group[f'{col}_outlier'] = ~group[col].between(lower,upper)
        out_list.append(group)
    return pd.concat(out_list, ignore_index=True)
    
df_label = detect_outlier(df_label)


#### Remove Outliers

In [ ]:
#Check outliers if they are actually outliers
#blood_pressure_diastolic_m: No outliers
#blood_pressure_mmHg_ No outliers
#pulse_1_min
df_label.loc[(df_label.index == 60) & (df_label['study_id'] == 5), 'pulse_1_min'] = np.nan #set one identified outlier (pulse = 20 bpm) to NaN
#vena_cava_diameter_mm: No outliers
#capillary_refill_time: No outliers
# walk_time: No outliers
# walk_speed: No outliers
# vas_score: No outliers


#### Remove Unnecessary Columns

In [ ]:
#Remove all _outlier columns
df_label = df_label.drop(columns=[col for col in df_label.columns if col.endswith('_outlier')])
#remove unnecessary columns
unnecessary_columns = ['redcap_event_name', 'redcap_repeat_instrument', 'redcap_repeat_instance', 'date_enrolled', 'dob', 'comments',
                       'demographics_complete',	'ae_dateevent',	'ae_description', 'notes_define_other_event',
                       'ae_unexpected',	'ae_unexpectedcomment',	'ae_related',	'ae_reschrisk',	'ae_serious',
                       'ae_reportable',	'ae_justification',	'ae_completedby', 'adverse_event_complete',	
                       'medication', 'icd10_diagnosis',	'wound_documentation', 'fall_documentation',
                       'mobilization_plan',	'fluid_balance_intake',	'fever', 'caritas_protocol_information_complete',
                       'dehydration_assessment_complete', 'unintentional_weight_loss',	'in_the_past_month_on_the_a',	
                       'if_yes_have_you_been_feeli', 'weak', 'frequency_weak','using_the_scale_below_plea',	
                       'low_activity_level', 'weakness', 'frailtyassessment_complete',
                       'vas_complete',	'study_comments', 'complete_study',	'withdraw_date', 'date_visit_4',
                       'withdraw_reason', 'completion_data_complete', 'promis_pac_m_009r1_0dfa17', 'promis_pac_m_105r1_2d14a6',
                       'promis_pac_m_002r1_f9884e',	'promis_pac_m_008r1_f1fe3b', 'promis_ped_sf_v10_physical_activity_4a_no_survey_complete', 'walk_time', 'walk_speed']
df_label = df_label.drop(columns=unnecessary_columns)



#### Fill Constants

In [ ]:
# Fill Weight and Height for every row
df_label[['age','gender', 'height', 'weight', 'bmi']] = df_label.groupby('study_id')[['age','gender', 'height', 'weight', 'bmi']].ffill().bfill()

#### Find Missing Dates

In [ ]:
# first cut out erollment date and last visit row
#drop enrollment and final visit
df_label = df_label[df_label.groupby('study_id').cumcount()>=2].reset_index(drop=True)
# impute correct dates for missing ones
df_label.loc[1, 'date'] = "2024-02-21"
df_label.loc[134, 'date'] = "2024-10-18"

In [ ]:
# Save state before Pain section modifies df and df_label
# (Dehydration section needs the original df with window_start, time_window, etc.)
df_base = df.copy()
df_label_base = df_label.copy()

### 2. Pain

#### Impute Missing Values

In [ ]:
#impute missing values depending on type of parameter
frequ_param = ['pulse_1_min', 'blood_pressure_mmhg', 'blood_pressure_diastolic_m', 'diarrhea', 'dry_muscous_membranes', 'is_the_skin_turgor_normal', 'pain_medication', 'self_reported_thirst', 'capillary_refill_time', 'vas_score']

fill_frequ_param = df_label.groupby('study_id')[frequ_param].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA).reset_index() #imputes most frequent values per ID
for col in frequ_param:
    df_label[col] = df_label[col].fillna(df_label['study_id'].map(fill_frequ_param.set_index('study_id')[col]))

df_label.isna().sum()


#### 2.1 Create Pain Labels

In [ ]:
# define Pain Label based on vas_score and pain_medication
# max. 10 points
# if >6 pain = 1 otherwise 0

def pain_label(df_label):
    pain_threshold = 5 #5
    pain_label =[]

    for id_, row in df_label.iterrows():
        pain_score = row['vas_score'] #virtual analog scale
        
        if row['pain_medication'] == 1:
            pain_score = np.minimum(pain_score + 1 , 10) #+3 (conservative bonus)
            
        if pain_score > pain_threshold: #>=
            pain_label.append(1)
        else:
            pain_label.append(0)

    return pain_label

df_label['pain_label'] = pain_label(df_label)

In [ ]:
df_label['pain_label'].value_counts()

# pain_label
# 0    102
# 1     57
# Name: count, dtype: int64


#### Merge df and df_label

In [ ]:
# rename ID column of df_label
df_label = df_label.rename(columns={'study_id': 'ID'})

#make sure date columns are datetime
df['date'] = df['window_start'].dt.date
df_label['date'] = pd.to_datetime(df_label['date']).dt.date

df = df.merge(
    df_label[['ID', 'date', 'pain_label']],
    on=['ID', 'date'],
    how='left'
)

df = df.drop(columns=['date'])

#### Cut Out Rows where pain_label is NaN

In [ ]:
df = df.dropna(subset=['pain_label']).reset_index(drop=True)

In [ ]:
df['pain_label'].value_counts()

In [ ]:
df.head(10)

#### 2.2 Pain Model

#### Find patients with changing label

In [ ]:
#find IDs where labels change
label_std = df.groupby('ID')['pain_label'].std()
pat_weights = label_std[label_std > 0].index.tolist() #get IDs where std is not 0 => labels change within patient -> give them more weight
pat_weights

#x_often_changes = 4

#label_changes = (df.groupby('ID')['pain_label'].apply(lambda x: x.diff().abs().sum()))
#pat_weights = label_changes.nlargest(x_often_changes).index.tolist()
#pat_weights


#### Find time windows for stronger weighting

In [ ]:
#define weight for visit datapoints
tw_weighting = 5

#take timw windows between 9 am and 11 am and weight them stronger
tw_visit = df['window_start'].dt.hour
tw_visit = tw_visit[(tw_visit >=9) & (tw_visit < 11)].index.tolist()
#generate default weights for all datapoints
tw_weights = pd.Series(1, index = df.index, dtype=float)
#fill in the visit weightings at the appropriate indices
tw_weights.loc[tw_visit] = tw_weighting


#### Prepare Data for Model

In [ ]:
df = df.drop(['time_window', 'window_start', 'window_end'], axis=1) #drop all parameter not measured by smartwatch

In [ ]:
df = df.drop(['Height_first', 'BodyMass_first'], axis=1, errors='ignore')

#### Split Data into Train and Test

In [ ]:
# V7: Multi-split evaluation — only test on patients where pain_label changes
from itertools import combinations

# ── Data-quality filter: exclude patients with too few samples ──
measurement_counts_pain = df.groupby('ID').size()
median_count_pain = measurement_counts_pain.median()
quality_threshold_pain = median_count_pain * 0.5
low_quality_ids_pain = measurement_counts_pain[measurement_counts_pain < quality_threshold_pain].index.tolist()

print(f"Per-patient sample counts:\n{measurement_counts_pain.sort_values().to_string()}")
print(f"\nMedian count: {median_count_pain:.0f}, threshold (50%): {quality_threshold_pain:.0f}")
print(f"Excluded (low quality): {low_quality_ids_pain}")

# Remove low-quality patients from the dataset used for LOPOCV
df_pain = df[~df['ID'].isin(low_quality_ids_pain)].copy()

label_std_pain = df_pain.groupby('ID')['pain_label'].std()
label_changing_ids_pain = sorted(label_std_pain[label_std_pain > 0].index.tolist())
all_ids_pain = sorted(df_pain.ID.unique().tolist())
print(f"\nAfter quality filter:")
print(f"Patients with changing pain labels: {label_changing_ids_pain} ({len(label_changing_ids_pain)}/{len(all_ids_pain)})")
print(f"All patient IDs: {all_ids_pain}")
print(f"Total samples remaining: {len(df_pain)}")

##### Drop features with almost no importance

In [ ]:
# Feature selection skipped in V7 — multi-split handles everything
print("Feature selection is handled inside multi-split loop (V7).")

#### Model

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# V7: Leave-One-Patient-Out CV (LOPOCV) — PAIN
# LOPO on label-changing patients; constant-label patients always train.
# Pools predictions across all folds for a single robust AUC.
# ═══════════════════════════════════════════════════════════════════════
import logging, os, json, joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, roc_curve,
)
from sklearn.model_selection import RandomizedSearchCV, cross_val_predict
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

PAIN_OUTPUT_DIR = 'Pain_V7_Results'
PARAMS_FILE = os.path.join(PAIN_OUTPUT_DIR, 'best_params.json')

print(f"LOPOCV on {len(label_changing_ids_pain)} label-changing patients: "
      f"{label_changing_ids_pain}")
print(f"Constant-label patients always in training: "
      f"{[x for x in all_ids_pain if x not in label_changing_ids_pain]}")

def build_pipeline(**xgb_params):
    """Build pipeline with optional fixed XGB parameters."""
    return Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", XGBClassifier(
            random_state=42,
            eval_metric='logloss',
            **xgb_params,
        )),
    ])

# ══════════════════════════════════════════════════════════════════════
# PHASE 1 — Load saved params or run optimisation & save
# ══════════════════════════════════════════════════════════════════════
if os.path.exists(PARAMS_FILE):
    with open(PARAMS_FILE) as f_:
        best_xgb_params = json.load(f_)
    print(f"\n  Loaded optimised params from {PARAMS_FILE}")
else:
    print("\n" + "=" * 70)
    print("  PHASE 1: Hyperparameter optimisation for LOPOCV (n_iter=1000)")
    print("=" * 70)

    param_grid_narrow = {
        'classifier__n_estimators':      [30, 40, 50, 60, 75, 100, 125, 150, 200],
        'classifier__max_depth':         [2, 3, 4, 5],
        'classifier__learning_rate':     [0.001, 0.003, 0.005, 0.008, 0.01, 0.02, 0.05, 0.1],
        'classifier__subsample':         [0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.9],
        'classifier__colsample_bytree':  [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
        'classifier__min_child_weight':  [1, 2, 3, 5, 7],
        'classifier__gamma':             [0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.5],
        'classifier__reg_alpha':         [0, 0.01, 0.1, 0.5, 1.0, 2.0],
        'classifier__reg_lambda':        [0.5, 1.0, 2.0, 3.0, 5.0],
        'classifier__scale_pos_weight':  [1, 1.5, 2, 3],
    }

    # Search using LOPO-style inner CV: hold out 1 label-changing patient,
    # use the rest for 3-fold CV search
    search_test_id = label_changing_ids_pain[0]
    search_train_ids = [x for x in all_ids_pain if x != search_test_id]
    search_train = df_pain[df_pain['ID'].isin(search_train_ids)]
    X_s = search_train.drop(columns=['pain_label', 'ID'])
    y_s = search_train['pain_label']
    pw_s = {i: (3.0 if i in label_changing_ids_pain else 1.0) for i in search_train_ids}
    sw_s = search_train['ID'].map(pw_s) * tw_weights.loc[search_train.index]

    global_search = RandomizedSearchCV(
        estimator=build_pipeline(),
        param_distributions=param_grid_narrow,
        scoring="roc_auc", cv=3, n_iter=1000,
        random_state=42, n_jobs=-1, verbose=0,
    )
    global_search.fit(X_s, y_s, classifier__sample_weight=sw_s)

    best_params_raw = global_search.best_params_
    best_xgb_params = {k.replace('classifier__', ''): v for k, v in best_params_raw.items()}

    print(f"\n  Best CV AUC: {global_search.best_score_:.4f}")
    print(f"  Configuration:")
    for k, v in sorted(best_xgb_params.items()):
        print(f"    {k}: {v}")

    os.makedirs(PAIN_OUTPUT_DIR, exist_ok=True)
    with open(PARAMS_FILE, 'w') as f_:
        json.dump(best_xgb_params, f_, indent=4)
    print(f"  Saved to {PARAMS_FILE}")

print(f"\n  Hyperparameters:")
for k, v in sorted(best_xgb_params.items()):
    print(f"    {k}: {v}")

# ══════════════════════════════════════════════════════════════════════
# PHASE 2 — LOPOCV on label-changing patients only
# Each fold: hold out 1 label-changing patient, train on all others
# ══════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("  LOPOCV: Leave-One-Patient-Out (label-changing patients)")
print("=" * 70)

all_pain_metrics = []
all_pain_importances = {}
best_pain_auc = -1
pain_model = None

# Pooled predictions
pooled_y_true = []
pooled_y_proba = []
pooled_y_pred = []

# ROC data
mean_fpr_grid = np.linspace(0, 1, 200)
all_pain_test_tprs = []
all_pain_cv_tprs = []
all_pain_cv_aucs = []
all_pain_train_metrics = []

for fold_idx, test_patient in enumerate(label_changing_ids_pain):
    train_ids = [x for x in all_ids_pain if x != test_patient]

    test_set = df_pain[df_pain['ID'] == test_patient]
    train_set = df_pain[df_pain['ID'].isin(train_ids)]

    X_tr = train_set.drop(columns=['pain_label', 'ID'])
    y_tr = train_set['pain_label']
    X_te = test_set.drop(columns=['pain_label', 'ID'])
    y_te = test_set['pain_label']

    # Sample weights
    patient_w = {i: (3.0 if i in label_changing_ids_pain else 1.0) for i in train_ids}
    sw = train_set['ID'].map(patient_w) * tw_weights.loc[train_set.index]

    # Train
    model = build_pipeline(**best_xgb_params)
    model.fit(X_tr, y_tr, classifier__sample_weight=sw)

    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    # Pool predictions
    pooled_y_true.extend(y_te.tolist())
    pooled_y_proba.extend(y_proba.tolist())
    pooled_y_pred.extend(y_pred.tolist())

    # Train metrics
    y_pred_tr = model.predict(X_tr)
    train_acc = accuracy_score(y_tr, y_pred_tr)
    all_pain_train_metrics.append({'fold': fold_idx+1, 'patient': test_patient,
                                    'train_accuracy': train_acc})

    pat_auc = roc_auc_score(y_te, y_proba)
    label_dist = dict(y_te.value_counts().sort_index())

    m = {
        'fold': fold_idx + 1,
        'test_patient': test_patient,
        'n_samples': len(y_te),
        'label_0': int(label_dist.get(0.0, 0)),
        'label_1': int(label_dist.get(1.0, 0)),
        'Accuracy': accuracy_score(y_te, y_pred),
        'Precision': precision_score(y_te, y_pred, zero_division=0),
        'Recall': recall_score(y_te, y_pred, zero_division=0),
        'F1': f1_score(y_te, y_pred, zero_division=0),
        'AUC': pat_auc,
    }
    all_pain_metrics.append(m)

    # ROC data
    fpr_s, tpr_s, _ = roc_curve(y_te, y_proba)
    interp_tpr = np.interp(mean_fpr_grid, fpr_s, tpr_s)
    interp_tpr[0] = 0.0
    all_pain_test_tprs.append(interp_tpr)

    # CV ROC on train set
    try:
        y_cv_proba = cross_val_predict(model, X_tr, y_tr, cv=3, method='predict_proba',
                                        params={'classifier__sample_weight': sw})[:, 1]
        cv_fpr, cv_tpr, _ = roc_curve(y_tr, y_cv_proba)
        cv_interp = np.interp(mean_fpr_grid, cv_fpr, cv_tpr)
        cv_interp[0] = 0.0
        all_pain_cv_tprs.append(cv_interp)
        all_pain_cv_aucs.append(roc_auc_score(y_tr, y_cv_proba))
    except Exception as e:
        print(f"    CV ROC failed for fold {fold_idx+1}: {e}")

    # Feature importances
    fi = model['classifier'].feature_importances_
    for fn, fv in zip(X_tr.columns, fi):
        all_pain_importances.setdefault(fn, []).append(fv)

    # Track best fold for SHAP
    if pat_auc > best_pain_auc:
        best_pain_auc = pat_auc
        pain_model = model
        X_train, X_test = X_tr, X_te
        y_train, y_test = y_tr, y_te
        sample_weights_train = sw

    print(f"  Fold {fold_idx+1}/{len(label_changing_ids_pain)}: "
          f"Patient {test_patient:>2} (n={len(y_te):>3}, "
          f"0:{m['label_0']:>3}/1:{m['label_1']:>3}) → "
          f"AUC={pat_auc:.3f}, Acc={m['Accuracy']:.3f}, "
          f"Prec={m['Precision']:.3f}, Rec={m['Recall']:.3f}")

# ══════════════════════════════════════════════════════════════════════
# POOLED RESULTS
# ══════════════════════════════════════════════════════════════════════
pooled_y_true = np.array(pooled_y_true)
pooled_y_proba = np.array(pooled_y_proba)
pooled_y_pred = np.array(pooled_y_pred)

pooled_auc = roc_auc_score(pooled_y_true, pooled_y_proba)
pooled_acc = accuracy_score(pooled_y_true, pooled_y_pred)
pooled_prec = precision_score(pooled_y_true, pooled_y_pred, zero_division=0)
pooled_rec = recall_score(pooled_y_true, pooled_y_pred, zero_division=0)
pooled_f1 = f1_score(pooled_y_true, pooled_y_pred, zero_division=0)

pain_results_df = pd.DataFrame(all_pain_metrics)
mean_pat_auc = pain_results_df['AUC'].mean()
std_pat_auc = pain_results_df['AUC'].std()

print(f"\n{'='*70}")
print(f"  PAIN — LOPOCV Results")
print(f"  {len(label_changing_ids_pain)} folds, "
      f"{len(pooled_y_true)} pooled test samples")
print(f"{'='*70}")
print(f"\n  ── Pooled metrics (all test predictions combined) ──")
print(f"  {'AUC':>12s}: {pooled_auc:.4f}")
print(f"  {'Accuracy':>12s}: {pooled_acc:.4f}")
print(f"  {'Precision':>12s}: {pooled_prec:.4f}")
print(f"  {'Recall':>12s}: {pooled_rec:.4f}")
print(f"  {'F1':>12s}: {pooled_f1:.4f}")
print(f"\n  ── Per-patient AUC ──")
print(f"  {'Mean':>12s}: {mean_pat_auc:.4f} ± {std_pat_auc:.4f}")
for _, row in pain_results_df.iterrows():
    print(f"    Patient {int(row['test_patient']):>2}: "
          f"AUC={row['AUC']:.3f}, Acc={row['Accuracy']:.3f} "
          f"(n={int(row['n_samples'])}, 0:{int(row['label_0'])}/1:{int(row['label_1'])})")

print(f"\n  ── Confusion matrix (pooled) ──")
cm = confusion_matrix(pooled_y_true, pooled_y_pred)
print(f"    TN={cm[0,0]:>4}  FP={cm[0,1]:>4}")
print(f"    FN={cm[1,0]:>4}  TP={cm[1,1]:>4}")

# Pooled ROC for plotting
pooled_fpr, pooled_tpr, _ = roc_curve(pooled_y_true, pooled_y_proba)

# Feature importances
pain_feature_impo_dict = {k: float(np.mean(v)) for k, v in all_pain_importances.items()}

# Save
os.makedirs(PAIN_OUTPUT_DIR, exist_ok=True)
pain_results_df.to_csv(os.path.join(PAIN_OUTPUT_DIR, 'lopo_per_patient.csv'), index=False)

avg_m = {
    'pooled_AUC': float(pooled_auc),
    'pooled_Accuracy': float(pooled_acc),
    'pooled_Precision': float(pooled_prec),
    'pooled_Recall': float(pooled_rec),
    'pooled_F1': float(pooled_f1),
    'per_patient_mean_AUC': float(mean_pat_auc),
    'per_patient_std_AUC': float(std_pat_auc),
}
with open(os.path.join(PAIN_OUTPUT_DIR, 'test_metrics.json'), 'w') as f_:
    json.dump(avg_m, f_, indent=4)

print(f"\n  Results saved to {PAIN_OUTPUT_DIR}/")


#### Training

In [ ]:
# ── Averaged Training Metrics (across all splits) — PAIN ──
_train_df = pd.DataFrame(all_pain_train_metrics)
print(f"PAIN — Averaged Training Accuracy ({len(_train_df)} splits)")
print(f"  Train Accuracy: {_train_df['train_accuracy'].mean():.4f} ± {_train_df['train_accuracy'].std():.4f}")
print(f"\n  (Best-split model training accuracy shown for reference)")
y_pred_train = pain_model.predict(X_train)
print(f"  Best-split Train Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")

#### Test

In [ ]:
# ── Averaged Test Metrics (across all splits) — PAIN ──
print(f"PAIN — Averaged Test Metrics ({len(pain_results_df)} splits)")
print("=" * 60)
for col in ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']:
    vals = pain_results_df[col]
    print(f"  {col:>12s}: {vals.mean():.4f} ± {vals.std():.4f}")
print("=" * 60)
print(f"\n  (Best-split test accuracy for reference)")
predictions = pain_model.predict(X_test)
print(f"  Best-split Test Accuracy: {accuracy_score(y_test, predictions):.4f}")

#### Feature Importance

In [ ]:
# ── Plot averaged feature importances (across all multi-splits) ── Pain
feature_names_list = list(pain_feature_impo_dict.keys())
importance_values = np.array([pain_feature_impo_dict[f] for f in feature_names_list])
sorted_indices = np.argsort(importance_values)[::-1]

plt.figure(figsize=(12, 5))
plt.bar(range(len(importance_values)), importance_values[sorted_indices], align="center")
plt.xticks(range(len(importance_values)),
           np.array(feature_names_list)[sorted_indices], rotation=90, fontsize=7)
plt.xlabel("Feature")
plt.ylabel("Avg Importance (across splits)")
plt.title('Pain — Averaged Feature Importance (Multi-Split)')
os.makedirs(PAIN_OUTPUT_DIR, exist_ok=True)
plt.savefig(os.path.join(PAIN_OUTPUT_DIR, 'Feature_importance_pain.tif'),
            format='tiff', dpi=300, bbox_inches='tight')
plt.show()

print(f"Top-10 features (Pain):")
for i in sorted_indices[:10]:
    print(f"  {feature_names_list[i]:40s}  {importance_values[i]:.6f}")

In [ ]:
pain_feature_impo_dict

#### T-Test: Top Features vs Pain Label

In [ ]:
# ── T-Test: Are the top features significantly different between pain=0 and pain=1? ──
from scipy.stats import ttest_ind

# df still contains the pain dataframe at this point (df_pain_final is set later)
_df_pain = df.copy()

# Get top features by averaged importance
_top_n = 10
_sorted_feats = sorted(pain_feature_impo_dict.items(), key=lambda x: x[1], reverse=True)
_top_feats = [f for f, _ in _sorted_feats[:_top_n]]

print("=" * 80)
print(f"  T-TEST: Top-{_top_n} Pain Features — Label 0 vs Label 1")
print("=" * 80)
print(f"  {'Feature':<40s}  {'Mean(0)':>9s}  {'Mean(1)':>9s}  {'t-stat':>8s}  {'p-value':>10s}  {'Sig?':>5s}")
print("-" * 80)

ttest_results_pain = []
for feat in _top_feats:
    if feat not in _df_pain.columns:
        continue
    grp0 = _df_pain.loc[_df_pain['pain_label'] == 0, feat].dropna()
    grp1 = _df_pain.loc[_df_pain['pain_label'] == 1, feat].dropna()
    if len(grp0) < 2 or len(grp1) < 2:
        continue
    t_stat, p_val = ttest_ind(grp0, grp1, equal_var=False)
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
    print(f"  {feat:<40s}  {grp0.mean():9.4f}  {grp1.mean():9.4f}  {t_stat:8.3f}  {p_val:10.2e}  {sig:>5s}")
    ttest_results_pain.append({'Feature': feat, 'Mean_0': grp0.mean(), 'Mean_1': grp1.mean(),
                               't_stat': t_stat, 'p_value': p_val, 'significant': p_val < 0.05})

print("-" * 80)
n_sig = sum(1 for r in ttest_results_pain if r['significant'])
print(f"  {n_sig}/{len(ttest_results_pain)} features significantly different at α=0.05")

# Save t-test results
ttest_pain_df = pd.DataFrame(ttest_results_pain)
ttest_pain_df.to_csv(os.path.join(PAIN_OUTPUT_DIR, 'ttest_pain_features.csv'), index=False)
print(f"  Saved to {PAIN_OUTPUT_DIR}/ttest_pain_features.csv")

#### AUC ROC 1

In [ ]:
# ── Averaged Test ROC (across all splits) — PAIN ──
from sklearn.metrics import auc

if len(all_pain_test_tprs) > 0:
    test_tprs = np.array(all_pain_test_tprs)
    mean_tpr_test = test_tprs.mean(axis=0)
    mean_tpr_test[-1] = 1.0
    mean_auc_test = auc(mean_fpr_grid, mean_tpr_test)
    std_tpr_test = test_tprs.std(axis=0)

    plt.figure(figsize=(8, 6))
    # Individual split curves (faint)
    for i, tpr_i in enumerate(test_tprs):
        auc_i = auc(mean_fpr_grid, tpr_i)
        plt.plot(mean_fpr_grid, tpr_i, alpha=0.15, linewidth=0.8)
    # Mean curve
    plt.plot(mean_fpr_grid, mean_tpr_test, color='b', linewidth=2,
             label=f'Mean Test ROC (AUC = {mean_auc_test:.3f} ± {np.std([auc(mean_fpr_grid, t) for t in test_tprs]):.3f})')
    # ±1 SD band
    tpr_upper = np.minimum(mean_tpr_test + std_tpr_test, 1)
    tpr_lower = np.maximum(mean_tpr_test - std_tpr_test, 0)
    plt.fill_between(mean_fpr_grid, tpr_lower, tpr_upper, alpha=0.2, color='b', label='± 1 SD')
    plt.plot([0, 1], [0, 1], 'k--', label='Chance')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Pain — Averaged Test ROC ({len(test_tprs)} splits)')
    plt.legend(loc='lower right')
    os.makedirs(PAIN_OUTPUT_DIR, exist_ok=True)
    plt.savefig(os.path.join(PAIN_OUTPUT_DIR, 'ROC_Curve_pain_averaged.tif'),
                format='tiff', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Mean Test AUC: {mean_auc_test:.4f}")
else:
    print("No test ROC data collected.")

#### AUC ROC 2

In [ ]:
# ── Averaged CV ROC (across all splits) — PAIN ──
if len(all_pain_cv_tprs) > 0:
    cv_tprs = np.array(all_pain_cv_tprs)
    mean_tpr_cv = cv_tprs.mean(axis=0)
    mean_tpr_cv[-1] = 1.0
    mean_auc_cv = auc(mean_fpr_grid, mean_tpr_cv)
    std_tpr_cv = cv_tprs.std(axis=0)

    plt.figure(figsize=(8, 6))
    # Individual split CV curves (faint)
    for i, tpr_i in enumerate(cv_tprs):
        plt.plot(mean_fpr_grid, tpr_i, alpha=0.15, linewidth=0.8)
    # Mean curve
    cv_aucs_arr = np.array(all_pain_cv_aucs)
    plt.plot(mean_fpr_grid, mean_tpr_cv, color='darkorange', linewidth=2,
             label=f'Mean CV ROC (AUC = {cv_aucs_arr.mean():.3f} ± {cv_aucs_arr.std():.3f})')
    # ±1 SD band
    cv_upper = np.minimum(mean_tpr_cv + std_tpr_cv, 1)
    cv_lower = np.maximum(mean_tpr_cv - std_tpr_cv, 0)
    plt.fill_between(mean_fpr_grid, cv_lower, cv_upper, alpha=0.2, color='darkorange', label='± 1 SD')
    plt.plot([0, 1], [0, 1], 'k--', label='Chance')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Pain — Averaged CV ROC ({len(cv_tprs)} splits)')
    plt.legend(loc='lower right')
    os.makedirs(PAIN_OUTPUT_DIR, exist_ok=True)
    plt.savefig(os.path.join(PAIN_OUTPUT_DIR, 'ROC_Curve_pain_CV_averaged.tif'),
                format='tiff', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Mean CV AUC: {cv_aucs_arr.mean():.4f} ± {cv_aucs_arr.std():.4f}")

    # ── Also show combined plot: CV vs Test ──
    pain_y_pred_proba = pain_model.predict_proba(X_test)[:, 1]  # keep for PR cell
else:
    print("No CV ROC data collected.")
    pain_y_pred_proba = pain_model.predict_proba(X_test)[:, 1]

#### Shape

In [ ]:
def calc_shap(model, X_test, label_name, output_dir):
    explainer = shap.Explainer(model.predict, X_test)
    shap_values = explainer(X_test)

    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, f"Shap_values_{label_name}.tif")
    shap.summary_plot(shap_values, X_test, show=False)
    plt.savefig(output_path, format='tiff', dpi=300, bbox_inches='tight')
    plt.show()

# NOTE: SHAP uses the best-performing split model (SHAP values don't average well across different models)
print(f"SHAP analysis uses best-split model (Test AUC={best_pain_auc:.4f})")
calc_shap(pain_model, X_test, 'pain', PAIN_OUTPUT_DIR)

#### Precision Recall

In [ ]:
def calc_precision_recall(y_test, y_pred_proba, label_name, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    y_test_np       = np.asarray(y_test).astype(int)
    y_pred_proba_np = np.asarray(y_pred_proba, dtype=float)

    # Full-test PR
    prec_full, rec_full, thr_full = precision_recall_curve(y_test_np, y_pred_proba_np)
    ap_full = average_precision_score(y_test_np, y_pred_proba_np)

    # Best-F1
    if len(thr_full) > 0:
        f1s = (2 * prec_full[:-1] * rec_full[:-1]) / np.clip(prec_full[:-1] + rec_full[:-1], 1e-12, None)
        i_best = int(np.nanargmax(f1s))
        best_thr  = float(thr_full[i_best])
        best_prec = float(prec_full[i_best])
        best_rec  = float(rec_full[i_best])
        best_f1   = float(f1s[i_best])
    else:
        best_thr, best_prec, best_rec, best_f1 = 0.5, float(prec_full[0]), float(rec_full[0]), 0.0

    # Diagnostics
    uniq_scores = np.unique(np.round(y_pred_proba_np, 6))
    if uniq_scores.size < 5:
        print(f"Warning: Only {uniq_scores.size} unique probability values.")

    # Bootstrap CI
    n_bootstraps = 100
    rng = np.random.RandomState(42)
    interp_curves = []
    mean_recall = np.linspace(0, 1, 200)

    for i in range(n_bootstraps):
        idx = rng.randint(0, len(y_test_np), len(y_test_np))
        y_s = y_test_np[idx]
        p_s = y_pred_proba_np[idx]
        if np.unique(y_s).size < 2:
            continue
        p, r, _ = precision_recall_curve(y_s, p_s)
        p_env = np.maximum.accumulate(p[::-1])[::-1]
        interp_curves.append(np.interp(mean_recall, r, p_env, left=p_env[0], right=p_env[-1]))

    interp_curves = np.vstack(interp_curves) if len(interp_curves) else None
    if interp_curves is not None:
        p_lower = np.percentile(interp_curves, 5, axis=0)
        p_upper = np.percentile(interp_curves, 95, axis=0)

    pos_rate = float(y_test_np.mean())

    # Plot
    sns.set_style("ticks"); sns.set_context("notebook")
    plt.figure(figsize=(8.5, 6.5))

    if interp_curves is not None:
        plt.fill_between(mean_recall, p_lower, p_upper, alpha=0.18, label="90% band", zorder=1)

    plt.step(rec_full, prec_full, where="post",
             linewidth=2.0, label=f"PR (AP={ap_full:.3f})", zorder=3)

    plt.scatter([best_rec], [best_prec], s=60, zorder=4,
                label=f"best F1={best_f1:.3f} @ thr={best_thr:.3f}")

    plt.hlines(pos_rate, 0, 1, linestyles="dashed", label=f"Baseline={pos_rate:.3f}", zorder=2)

    plt.xlim(0, 1); plt.ylim(0, 1.02)
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title(f"Precision-Recall for {label_name} (best-split model)")
    plt.legend(loc="lower left")
    plt.tight_layout()

    out_path = os.path.join(output_dir, f"pr_curve_{label_name}.png")
    plt.savefig(out_path, dpi=150); plt.show(); plt.close()
    print(f"PR curve saved to {out_path}")

# NOTE: PR curve uses best-split model data (bootstrapped CI from single best split)
print(f"PR curve uses best-split model (Test AUC={best_pain_auc:.4f})")
calc_precision_recall(y_test, pain_y_pred_proba, 'pain', PAIN_OUTPUT_DIR)

In [ ]:
# ── Save Pain results for publication section ──
X_test_pain = X_test.copy()
y_test_pain = y_test.copy()
X_train_pain = X_train.copy()
y_train_pain = y_train.copy()
pain_preds = predictions.copy()
pain_weights = sample_weights_train.copy()
# df is modified in dehydration section, save pain df
df_pain_final = df.copy()
print("Pain results saved.")

### 3. Dehydration

#### Impute Missing Values

In [ ]:
# Restore original df and df_label before Dehydration processing
# (Pain section dropped columns and modified df_label)
df = df_base.copy()
df_label = df_label_base.copy()

In [ ]:
#impute missing values depending on type of parameter
frequ_param = ['pulse_1_min', 'blood_pressure_mmhg', 'blood_pressure_diastolic_m', 'diarrhea', 'dry_muscous_membranes', 'is_the_skin_turgor_normal', 'pain_medication', 'self_reported_thirst', 'capillary_refill_time', 'vas_score']

fill_frequ_param = df_label.groupby('study_id')[frequ_param].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA).reset_index() #imputes most frequent values per ID
for col in frequ_param:
    df_label[col] = df_label[col].fillna(df_label['study_id'].map(fill_frequ_param.set_index('study_id')[col]))

df_label.isna().sum()

#### 3.1 Create Dehydration Labels

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Dehydration Label V3 — Patient-Relative Z-Score + Correlation-Weighted
# ═══════════════════════════════════════════════════════════════════════
# V2 used binary above/below patient mean → too noisy (AUC 0.46).
# V3 uses z-scores for continuous signs (captures magnitude of deviation)
# and weights based on sub-label t-test correlation with Watch features:
#
#   Sub-label                   Watch correlation  Weight
#   capillary_refill_time       13/15              3.0
#   blood_pressure_mmhg         12/15              2.5
#   self_reported_thirst        12/15              2.5
#   dry_muscous_membranes       12/15              2.5
#   diarrhea                    11/15              2.0
#   vena_cava_diameter_mm       10/15              2.0
#   is_the_skin_turgor_normal    9/15              1.5
#   pulse_1_min                  9/15              1.5
# ═══════════════════════════════════════════════════════════════════════

def dehydration_label(df_label, threshold=2.5):
    """Patient-relative dehydration scoring using z-scores.
    
    Continuous signs: z-score within patient (how many SDs worse than 
    their personal mean). Only positive z-scores contribute (i.e. only
    deviations TOWARD dehydration add to the score).
    
    Binary signs: weighted presence (1 if present, 0 if not).
    Weights set by how well the Watch can detect each sign (from t-tests).
    """
    df = df_label.copy()
    
    # ── Patient z-scores for continuous signs ──
    # Higher z = more dehydrated relative to patient mean
    
    # Capillary refill: higher = worse → z = (val - mean) / std
    df['crt_mean'] = df.groupby('study_id')['capillary_refill_time'].transform('mean')
    df['crt_std']  = df.groupby('study_id')['capillary_refill_time'].transform('std')
    df['crt_std']  = df['crt_std'].replace(0, np.nan)  # avoid div by 0 for constant patients
    df['crt_z']    = ((df['capillary_refill_time'] - df['crt_mean']) / df['crt_std']).clip(lower=0).fillna(0)
    
    # Blood pressure: lower = worse → z = (mean - val) / std
    df['bp_mean'] = df.groupby('study_id')['blood_pressure_mmhg'].transform('mean')
    df['bp_std']  = df.groupby('study_id')['blood_pressure_mmhg'].transform('std')
    df['bp_std']  = df['bp_std'].replace(0, np.nan)
    df['bp_z']    = ((df['bp_mean'] - df['blood_pressure_mmhg']) / df['bp_std']).clip(lower=0).fillna(0)
    
    # Pulse: higher = worse → z = (val - mean) / std
    df['pulse_mean'] = df.groupby('study_id')['pulse_1_min'].transform('mean')
    df['pulse_std']  = df.groupby('study_id')['pulse_1_min'].transform('std')
    df['pulse_std']  = df['pulse_std'].replace(0, np.nan)
    df['pulse_z']    = ((df['pulse_1_min'] - df['pulse_mean']) / df['pulse_std']).clip(lower=0).fillna(0)
    
    # Vena cava: smaller = worse → z = (mean - val) / std
    df['vc_mean'] = df.groupby('study_id')['vena_cava_diameter_mm'].transform('mean')
    df['vc_std']  = df.groupby('study_id')['vena_cava_diameter_mm'].transform('std')
    df['vc_std']  = df['vc_std'].replace(0, np.nan)
    df['vc_z']    = ((df['vc_mean'] - df['vena_cava_diameter_mm']) / df['vc_std']).clip(lower=0).fillna(0)
    
    # ── Weighted score ──
    scores = pd.Series(0.0, index=df.index)
    
    # Continuous signs: z-score * weight (proportional to Watch correlation)
    scores += df['crt_z']   * 3.0    # capillary_refill_time: 13/15
    scores += df['bp_z']    * 2.5    # blood_pressure_mmhg:   12/15
    scores += df['pulse_z'] * 1.5    # pulse_1_min:            9/15
    scores += df['vc_z']    * 2.0    # vena_cava_diameter_mm: 10/15
    
    # Binary signs: presence * weight (proportional to Watch correlation)
    scores += (df['self_reported_thirst'] != 0).astype(float)      * 2.5  # 12/15
    scores += (df['dry_muscous_membranes'] != 0).astype(float)     * 2.5  # 12/15
    scores += (df['diarrhea'] != 0).astype(float)                  * 2.0  # 11/15
    scores += (df['is_the_skin_turgor_normal'] != 1).astype(float) * 1.5  #  9/15
    
    # ── Apply threshold ──
    labels = (scores > threshold).astype(int).tolist()
    
    # ── Print distribution ──
    print(f"  Dehydration label V3 (z-score + correlation-weighted, threshold={threshold}):")
    print(f"    Score range: {scores.min():.2f} – {scores.max():.2f}, "
          f"mean={scores.mean():.2f}, median={scores.median():.2f}")
    print(f"    Label=1 (dehydrated): {sum(labels)} ({sum(labels)/len(labels)*100:.1f}%)")
    print(f"    Label=0 (hydrated):   {len(labels)-sum(labels)} ({(len(labels)-sum(labels))/len(labels)*100:.1f}%)")
    
    # Show per-patient label change
    _tmp = pd.DataFrame({'study_id': df['study_id'], 'label': labels})
    _changes = _tmp.groupby('study_id')['label'].nunique()
    n_changing = (_changes > 1).sum()
    print(f"    Patients with changing labels: {n_changing}/{len(_changes)}")
    
    # Show score components for debugging
    print(f"\n    Score components (mean contribution):")
    print(f"      capillary_refill z*1.0:  {(df['crt_z'] * 1.0).mean():.3f}")
    print(f"      blood_pressure  z*3:   {(df['bp_z'] * 3).mean():.3f}")
    print(f"      pulse           z*3:   {(df['pulse_z'] * 3).mean():.3f}")
    print(f"      vena_cava       z*4.0:   {(df['vc_z'] * 4.0).mean():.3f}")
    print(f"      thirst          *1.5:    {((df['self_reported_thirst'] != 0).astype(float) * 1.5).mean():.3f}")
    print(f"      dry_mucous      *1.5:    {((df['dry_muscous_membranes'] != 0).astype(float) * 1.5).mean():.3f}")
    print(f"      diarrhea        *3.0:    {((df['diarrhea'] != 0).astype(float) * 3.0).mean():.3f}")
    print(f"      skin_turgor     *1:    {((df['is_the_skin_turgor_normal'] != 1).astype(float) * 1).mean():.3f}")
    
    # Show threshold sensitivity
    print(f"\n    Threshold sensitivity:")
    for t in [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]:
        n1 = (scores > t).sum()
        _tmp2 = pd.DataFrame({'study_id': df['study_id'], 'l': (scores > t).astype(int)})
        n_ch = (_tmp2.groupby('study_id')['l'].nunique() > 1).sum()
        print(f"      threshold={t:.1f}: {n1:3d} dehydrated ({n1/len(scores)*100:5.1f}%) — {n_ch} patients changing")
    
    return labels

df_label['dehydration_label'] = dehydration_label(df_label, threshold=2.5)

In [ ]:
df_label['dehydration_label'].value_counts()

#### Merge df and df_label

In [ ]:
# rename ID column of df_label
df_label = df_label.rename(columns={'study_id': 'ID'})

#make sure date columns are datetime
df['date'] = df['window_start'].dt.date
df_label['date'] = pd.to_datetime(df_label['date']).dt.date

df = df.merge(
    df_label[['ID', 'date', 'dehydration_label']],
    on=['ID', 'date'],
    how='left'
)

df = df.drop(columns=['date'])

#### Cut Out Rows where dehydration_label is NaN

In [ ]:
df = df.dropna(subset=['dehydration_label']).reset_index(drop=True)

In [ ]:
df['dehydration_label'].value_counts()

#### Find IDs where label changes

In [ ]:
#find IDs where labels change
#label_std = df.groupby('ID')['dehydration_label'].std()
#pat_weights = label_std[label_std > 0].index.tolist() #get IDs where std is not 0 => labels change within patient -> give them more weight
#pat_weights

x_often_changes = 4

label_changes = (df.groupby('ID')['dehydration_label'].apply(lambda x: x.diff().abs().sum()))
pat_weights = label_changes.nlargest(x_often_changes).index.tolist()
pat_weights

#### Find time windows for stronger weights

In [ ]:
#define weight for visit datapoints
tw_weighting = 1 #4

#take timw windows between 9 am and 11 pm and weight them stronger
tw_visit = df['window_start'].dt.hour
tw_visit = tw_visit[(tw_visit >=9) & (tw_visit < 11)].index.tolist()
#generate default weights for all datapoints
tw_weights = pd.Series(1, index = df.index, dtype=float)
#fill in the visit weightings at the appropriate indices
tw_weights.loc[tw_visit] = tw_weighting

#### 3.2 Dehydration Model

#### Prepare Data for Model

In [ ]:
df = df.drop(['time_window', 'window_start', 'window_end'], axis=1) #drop all parameter not needed for model

In [ ]:
df = df.drop(['Height_first', 'BodyMass_first'], axis=1)

#### Split Data into Train and Test

In [ ]:
# V7: Multi-split evaluation — only test on patients where dehydration_label changes
# AND who have sufficient data quality (>= 50% of median measurement count)

# ── Data-quality filter: exclude patients with too few samples ──
measurement_counts_dehy = df.groupby('ID').size()
median_count_dehy = measurement_counts_dehy.median()
quality_threshold_dehy = median_count_dehy * 0.5
low_quality_ids_dehy = measurement_counts_dehy[measurement_counts_dehy < quality_threshold_dehy].index.tolist()

print(f"Per-patient sample counts:\n{measurement_counts_dehy.sort_values().to_string()}")
print(f"\nMedian count: {median_count_dehy:.0f}, threshold (50%): {quality_threshold_dehy:.0f}")
print(f"Excluded (low quality): {low_quality_ids_dehy}")

# Remove low-quality patients from the dataset used for modelling
df_dehy = df[~df['ID'].isin(low_quality_ids_dehy)].copy()

label_std_dehy = df_dehy.groupby('ID')['dehydration_label'].std()
label_changing_ids_dehy = sorted(label_std_dehy[label_std_dehy > 0].index.tolist())
all_ids_dehy = sorted(df_dehy.ID.unique().tolist())
print(f"\nAfter quality filter:")
print(f"Patients with changing dehydration labels: {label_changing_ids_dehy} ({len(label_changing_ids_dehy)}/{len(all_ids_dehy)})")
print(f"All patient IDs: {all_ids_dehy}")
print(f"Total samples remaining: {len(df_dehy)}")

##### Drop Features with almost no importance

In [ ]:
# Feature selection skipped in V7 — multi-split handles everything
print("Feature selection is handled inside multi-split loop (V7).")

#### Model

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# V7: Multi-Split Random Forest — DEHYDRATION
# Only test on patients whose dehydration_label changes.
# Average metrics across all splits.
# ═══════════════════════════════════════════════════════════════════════
DEHY_OUTPUT_DIR = 'Dehydration_V7_Results'

# ── Generate test splits from ALL patients ──
n_test_d = min(N_TEST_PATIENTS, len(all_ids_dehy))
all_combos_dehy_raw = list(combinations(all_ids_dehy, n_test_d))
random.seed(43)
random.shuffle(all_combos_dehy_raw)
all_combos_dehy = all_combos_dehy_raw[:MAX_SPLITS]

print(f"Running {len(all_combos_dehy)} dehydration splits "
      f"(from {len(all_ids_dehy)} patients, test size {n_test_d})")

all_dehy_metrics = []
all_dehy_importances = {}
best_dehy_auc = -1
dehy_model = None

# ── ROC data collection for averaged curves ──
all_dehy_test_tprs = []       # interpolated test ROC per split
all_dehy_cv_tprs = []         # interpolated CV ROC per split
all_dehy_cv_aucs = []         # CV AUC per split
all_dehy_train_metrics = []   # train accuracy per split

for split_idx, test_ids in enumerate(all_combos_dehy):
    test_ids = list(test_ids)
    train_ids = [x for x in all_ids_dehy if x not in test_ids]

    test_set = df_dehy[df_dehy['ID'].isin(test_ids)]
    train_set = df_dehy[df_dehy['ID'].isin(train_ids)]

    X_tr = train_set.drop(columns=['dehydration_label', 'ID'])
    y_tr = train_set['dehydration_label']
    X_te = test_set.drop(columns=['dehydration_label', 'ID'])
    y_te = test_set['dehydration_label']

    # Sample weights
    weighted_ids = list(set(pat_weights) & set(train_ids))
    patient_w = {i: 1 for i in train_ids}
    sw = train_set['ID'].map(patient_w)
    tw_w = tw_weights.loc[train_set.index]
    sw = sw * tw_w

    if y_te.nunique() < 2:
        print(f"  Split {split_idx+1}: SKIP (test has only class {y_te.unique()[0]})")
        continue

    search = RandomizedSearchCV(
        estimator=build_pipeline(),
        param_distributions=param_grid,
        scoring="roc_auc", cv=3, n_iter=15,
        random_state=42, n_jobs=-1, verbose=0,
    )
    search.fit(X_tr, y_tr, classifier__sample_weight=sw)
    model = search.best_estimator_

    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    # ── Train metrics ──
    y_pred_tr = model.predict(X_tr)
    train_acc = accuracy_score(y_tr, y_pred_tr)
    all_dehy_train_metrics.append({'split': split_idx+1, 'train_accuracy': train_acc})

    m = {
        'split': split_idx + 1,
        'test_ids': test_ids,
        'cv_auc': search.best_score_,
        'Accuracy': accuracy_score(y_te, y_pred),
        'Precision': precision_score(y_te, y_pred, zero_division=0),
        'Recall': recall_score(y_te, y_pred, zero_division=0),
        'F1': f1_score(y_te, y_pred, zero_division=0),
        'AUC': roc_auc_score(y_te, y_proba),
    }
    all_dehy_metrics.append(m)

    # ── Collect test ROC curve data ──
    fpr_s, tpr_s, _ = roc_curve(y_te, y_proba)
    interp_tpr = np.interp(mean_fpr_grid, fpr_s, tpr_s)
    interp_tpr[0] = 0.0
    all_dehy_test_tprs.append(interp_tpr)

    # ── Collect CV ROC curve data ──
    try:
        y_cv_proba = cross_val_predict(model, X_tr, y_tr, cv=3, method='predict_proba',
                                        params={'classifier__sample_weight': sw})[:, 1]
        cv_fpr, cv_tpr, _ = roc_curve(y_tr, y_cv_proba)
        cv_interp = np.interp(mean_fpr_grid, cv_fpr, cv_tpr)
        cv_interp[0] = 0.0
        all_dehy_cv_tprs.append(cv_interp)
        all_dehy_cv_aucs.append(roc_auc_score(y_tr, y_cv_proba))
    except Exception as e:
        print(f"    CV ROC failed for split {split_idx+1}: {e}")
        all_dehy_cv_aucs.append(search.best_score_)

    fi = model['classifier'].feature_importances_
    for fn, fv in zip(X_tr.columns, fi):
        all_dehy_importances.setdefault(fn, []).append(fv)

    if m['AUC'] > best_dehy_auc:
        best_dehy_auc = m['AUC']
        dehy_model = model
        X_train, X_test = X_tr, X_te
        y_train, y_test = y_tr, y_te
        sample_weights_train = sw

    print(f"  Split {split_idx+1}/{len(all_combos_dehy)}: test={test_ids} → "
          f"CV={m['cv_auc']:.3f}, Test AUC={m['AUC']:.3f}, Train Acc={train_acc:.3f}")

# ── Aggregate ──
dehy_results_df = pd.DataFrame(all_dehy_metrics)
print(f"\n{'='*70}")
print(f"  DEHYDRATION — Multi-Split Results ({len(dehy_results_df)} splits)")
print(f"{'='*70}")
for col in ['cv_auc', 'Accuracy', 'Precision', 'Recall', 'F1', 'AUC']:
    vals = dehy_results_df[col]
    print(f"  {col:>12s}: {vals.mean():.4f} ± {vals.std():.4f}  "
          f"(min={vals.min():.3f}, max={vals.max():.3f})")

print(f"\n  Best split AUC={best_dehy_auc:.4f}")

# Averaged feature importances
dehy_feature_impo_dict = {k: float(np.mean(v)) for k, v in all_dehy_importances.items()}

# Save
os.makedirs(DEHY_OUTPUT_DIR, exist_ok=True)
dehy_results_df.to_csv(os.path.join(DEHY_OUTPUT_DIR, 'multi_split_results.csv'), index=False)

avg_m = {c: float(dehy_results_df[c].mean()) for c in ['Accuracy','Precision','Recall','F1','AUC']}
with open(os.path.join(DEHY_OUTPUT_DIR, 'test_metrics.json'), 'w') as f_:
    json.dump(avg_m, f_, indent=4)

print(f"  Results saved to {DEHY_OUTPUT_DIR}/")

#### Training

In [ ]:
# ── Averaged Training Metrics (across all splits) — DEHYDRATION ──
_train_df_d = pd.DataFrame(all_dehy_train_metrics)
print(f"DEHYDRATION — Averaged Training Accuracy ({len(_train_df_d)} splits)")
print(f"  Train Accuracy: {_train_df_d['train_accuracy'].mean():.4f} ± {_train_df_d['train_accuracy'].std():.4f}")
print(f"\n  (Best-split model training accuracy shown for reference)")
y_pred_train = dehy_model.predict(X_train)
print(f"  Best-split Train Accuracy: {accuracy_score(y_train, y_pred_train):.4f}")

#### Test

In [ ]:
# ── Averaged Test Metrics (across all splits) — DEHYDRATION ──
print(f"DEHYDRATION — Averaged Test Metrics ({len(dehy_results_df)} splits)")
print("=" * 60)
for col in ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']:
    vals = dehy_results_df[col]
    print(f"  {col:>12s}: {vals.mean():.4f} ± {vals.std():.4f}")
print("=" * 60)
print(f"\n  (Best-split test accuracy for reference)")
predictions = dehy_model.predict(X_test)
print(f"  Best-split Test Accuracy: {accuracy_score(y_test, predictions):.4f}")

#### Feature Importance

In [ ]:
# ── Plot averaged feature importances (across all multi-splits) ── Dehydration
feature_names_list_d = list(dehy_feature_impo_dict.keys())
importance_values_d = np.array([dehy_feature_impo_dict[f] for f in feature_names_list_d])
sorted_indices_d = np.argsort(importance_values_d)[::-1]

plt.figure(figsize=(12, 5))
plt.bar(range(len(importance_values_d)), importance_values_d[sorted_indices_d], align="center")
plt.xticks(range(len(importance_values_d)),
           np.array(feature_names_list_d)[sorted_indices_d], rotation=90, fontsize=7)
plt.xlabel("Feature")
plt.ylabel("Avg Importance (across splits)")
plt.title('Dehydration — Averaged Feature Importance (Multi-Split)')
os.makedirs(DEHY_OUTPUT_DIR, exist_ok=True)
plt.savefig(os.path.join(DEHY_OUTPUT_DIR, 'Feature_importance_dehydration.tif'),
            format='tiff', dpi=300, bbox_inches='tight')
plt.show()

print(f"Top-10 features (Dehydration):")
for i in sorted_indices_d[:10]:
    print(f"  {feature_names_list_d[i]:40s}  {importance_values_d[i]:.6f}")

In [ ]:
dehy_feature_impo_dict

#### T-Test: Top Features vs Dehydration Label

In [ ]:
# ── T-Test: Are the top features significantly different between dehydration=0 and dehydration=1? ──
from scipy.stats import ttest_ind

# df still contains the dehydration dataframe at this point (df_dehy_final is set later)
_df_dehy = df.copy()

# Get top features by averaged importance
_top_n_d = 10
_sorted_feats_d = sorted(dehy_feature_impo_dict.items(), key=lambda x: x[1], reverse=True)
_top_feats_d = [f for f, _ in _sorted_feats_d[:_top_n_d]]

print("=" * 80)
print(f"  T-TEST: Top-{_top_n_d} Dehydration Features — Label 0 vs Label 1")
print("=" * 80)
print(f"  {'Feature':<40s}  {'Mean(0)':>9s}  {'Mean(1)':>9s}  {'t-stat':>8s}  {'p-value':>10s}  {'Sig?':>5s}")
print("-" * 80)

ttest_results_dehy = []
for feat in _top_feats_d:
    if feat not in _df_dehy.columns:
        continue
    grp0 = _df_dehy.loc[_df_dehy['dehydration_label'] == 0, feat].dropna()
    grp1 = _df_dehy.loc[_df_dehy['dehydration_label'] == 1, feat].dropna()
    if len(grp0) < 2 or len(grp1) < 2:
        continue
    t_stat, p_val = ttest_ind(grp0, grp1, equal_var=False)
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
    print(f"  {feat:<40s}  {grp0.mean():9.4f}  {grp1.mean():9.4f}  {t_stat:8.3f}  {p_val:10.2e}  {sig:>5s}")
    ttest_results_dehy.append({'Feature': feat, 'Mean_0': grp0.mean(), 'Mean_1': grp1.mean(),
                                't_stat': t_stat, 'p_value': p_val, 'significant': p_val < 0.05})

print("-" * 80)
n_sig_d = sum(1 for r in ttest_results_dehy if r['significant'])
print(f"  {n_sig_d}/{len(ttest_results_dehy)} features significantly different at α=0.05")

# Save t-test results
ttest_dehy_df = pd.DataFrame(ttest_results_dehy)
ttest_dehy_df.to_csv(os.path.join(DEHY_OUTPUT_DIR, 'ttest_dehydration_features.csv'), index=False)
print(f"  Saved to {DEHY_OUTPUT_DIR}/ttest_dehydration_features.csv")

#### T-Test: Dehydration Sub-Labels vs Apple Watch Features

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# T-Test: Which dehydration sub-labels correlate with Apple Watch features?
# This tells us which clinical signs the Watch can actually "see".
# ═══════════════════════════════════════════════════════════════════════
from scipy.stats import ttest_ind

# ── Reconstruct merged dataframe: Watch features + clinical sub-labels ──
_df_watch = df_base.copy()
_df_watch['date'] = _df_watch['window_start'].dt.date
_df_clinical = df_label.copy()
_df_clinical['date'] = pd.to_datetime(_df_clinical['date']).dt.date

# Dehydration sub-label columns and how to binarize them
sub_label_defs = {
    'capillary_refill_time':      ('>=2',  lambda x: (x >= 2).astype(int)),
    'is_the_skin_turgor_normal':  ('abnormal (!=1)', lambda x: (x != 1).astype(int)),
    'diarrhea':                   ('present (!=0)',   lambda x: (x != 0).astype(int)),
    'blood_pressure_mmhg':        ('low (<=100)',     lambda x: (x <= 100).astype(int)),
    'pulse_1_min':                ('high (>91)',      lambda x: (x > 91).astype(int)),
    'dry_muscous_membranes':      ('present (!=0)',   lambda x: (x != 0).astype(int)),
    'self_reported_thirst':       ('present (!=0)',   lambda x: (x != 0).astype(int)),
    'vena_cava_diameter_mm':      ('<patient mean',   None),  # handled separately
}

# Binarize sub-labels
for col, (desc, binarizer) in sub_label_defs.items():
    if binarizer is not None:
        _df_clinical[f'{col}_bin'] = binarizer(_df_clinical[col])
    elif col == 'vena_cava_diameter_mm':
        vc_mean = _df_clinical.groupby('ID')['vena_cava_diameter_mm'].transform('mean')
        _df_clinical['vena_cava_diameter_mm_bin'] = (
            (_df_clinical['vena_cava_diameter_mm'] < vc_mean) & 
            _df_clinical['vena_cava_diameter_mm'].notna()
        ).astype(int)

# Merge Watch features + binarized sub-labels
bin_cols = [f'{c}_bin' for c in sub_label_defs]
_merged = _df_watch.merge(
    _df_clinical[['ID', 'date'] + bin_cols],
    on=['ID', 'date'], how='inner'
)

# ── Get top Watch features from model importance ──
_top_n = 15
_sorted_feats = sorted(dehy_feature_impo_dict.items(), key=lambda x: x[1], reverse=True)
_top_watch_feats = [f for f, _ in _sorted_feats[:_top_n] if f in _merged.columns]

# ── Run t-tests: for each sub-label, which Watch features differ? ──
print("=" * 100)
print("  T-TEST: Dehydration Sub-Labels vs Top Apple Watch Features")
print("  Question: Which clinical signs produce distinct Watch data patterns?")
print("=" * 100)

sublabel_sig_counts = {}

for col in sub_label_defs:
    bin_col = f'{col}_bin'
    desc = sub_label_defs[col][0]
    
    grp0 = _merged[_merged[bin_col] == 0]
    grp1 = _merged[_merged[bin_col] == 1]
    
    if len(grp1) < 3:
        print(f"\n  ⚠ {col} ({desc}): only {len(grp1)} positive cases — skipped")
        sublabel_sig_counts[col] = 0
        continue
    
    print(f"\n  ── {col} ({desc}): n=0→{len(grp0)}, n=1→{len(grp1)} ──")
    
    sig_count = 0
    results_for_sublabel = []
    
    for feat in _top_watch_feats:
        v0 = grp0[feat].dropna()
        v1 = grp1[feat].dropna()
        if len(v0) < 2 or len(v1) < 2:
            continue
        t_stat, p_val = ttest_ind(v0, v1, equal_var=False)
        sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
        if p_val < 0.05:
            sig_count += 1
        results_for_sublabel.append((feat, v0.mean(), v1.mean(), t_stat, p_val, sig))
    
    # Print only significant results + top 3 non-significant
    sig_results = [r for r in results_for_sublabel if r[5]]
    nonsig_results = [r for r in results_for_sublabel if not r[5]][:3]
    
    print(f"    {'Watch Feature':<42s}  {'Mean(0)':>9s}  {'Mean(1)':>9s}  {'t-stat':>8s}  {'p-value':>10s}  {'Sig':>4s}")
    print(f"    {'-'*85}")
    for feat, m0, m1, ts, pv, sg in sig_results:
        print(f"    {feat:<42s}  {m0:9.4f}  {m1:9.4f}  {ts:8.3f}  {pv:10.2e}  {sg:>4s}")
    if nonsig_results:
        print(f"    ... ({len([r for r in results_for_sublabel if not r[5]])} non-significant features omitted, showing top 3):")
        for feat, m0, m1, ts, pv, sg in nonsig_results:
            print(f"    {feat:<42s}  {m0:9.4f}  {m1:9.4f}  {ts:8.3f}  {pv:10.2e}  {sg:>4s}")
    
    sublabel_sig_counts[col] = sig_count

# ── Summary: which sub-labels have the most Watch-feature correlations? ──
print(f"\n{'='*100}")
print(f"  SUMMARY: Sub-label → # significant Watch feature associations (out of {len(_top_watch_feats)})")
print(f"{'='*100}")
sorted_sublabels = sorted(sublabel_sig_counts.items(), key=lambda x: x[1], reverse=True)
for col, n_sig in sorted_sublabels:
    desc = sub_label_defs[col][0]
    bar = '█' * n_sig + '░' * (len(_top_watch_feats) - n_sig)
    print(f"  {col:<35s} {desc:<20s}  {n_sig:>2d}/{len(_top_watch_feats)}  {bar}")

print(f"\n  → Sub-labels with MORE significant associations are better captured by Watch data")
print(f"    and should get HIGHER weights in the dehydration label definition.")

# Save results
_sublabel_results = pd.DataFrame([
    {'sub_label': col, 'description': sub_label_defs[col][0], 
     'n_significant': sublabel_sig_counts[col], 
     'n_total_features': len(_top_watch_feats)}
    for col in sub_label_defs
]).sort_values('n_significant', ascending=False)
_sublabel_results.to_csv(os.path.join(DEHY_OUTPUT_DIR, 'sublabel_ttest_summary.csv'), index=False)
print(f"\n  Saved to {DEHY_OUTPUT_DIR}/sublabel_ttest_summary.csv")

#### AUC ROC 1

In [ ]:
# ── Averaged Test ROC (across all splits) — DEHYDRATION ──
if len(all_dehy_test_tprs) > 0:
    test_tprs_d = np.array(all_dehy_test_tprs)
    mean_tpr_test_d = test_tprs_d.mean(axis=0)
    mean_tpr_test_d[-1] = 1.0
    mean_auc_test_d = auc(mean_fpr_grid, mean_tpr_test_d)
    std_tpr_test_d = test_tprs_d.std(axis=0)

    plt.figure(figsize=(8, 6))
    for i, tpr_i in enumerate(test_tprs_d):
        plt.plot(mean_fpr_grid, tpr_i, alpha=0.15, linewidth=0.8)
    plt.plot(mean_fpr_grid, mean_tpr_test_d, color='b', linewidth=2,
             label=f'Mean Test ROC (AUC = {mean_auc_test_d:.3f} ± {np.std([auc(mean_fpr_grid, t) for t in test_tprs_d]):.3f})')
    tpr_upper_d = np.minimum(mean_tpr_test_d + std_tpr_test_d, 1)
    tpr_lower_d = np.maximum(mean_tpr_test_d - std_tpr_test_d, 0)
    plt.fill_between(mean_fpr_grid, tpr_lower_d, tpr_upper_d, alpha=0.2, color='b', label='± 1 SD')
    plt.plot([0, 1], [0, 1], 'k--', label='Chance')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Dehydration — Averaged Test ROC ({len(test_tprs_d)} splits)')
    plt.legend(loc='lower right')
    os.makedirs(DEHY_OUTPUT_DIR, exist_ok=True)
    plt.savefig(os.path.join(DEHY_OUTPUT_DIR, 'ROC_Curve_dehydration_averaged.tif'),
                format='tiff', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Mean Test AUC: {mean_auc_test_d:.4f}")
else:
    print("No test ROC data collected.")

#### AUC ROC 2

In [ ]:
# ── Averaged CV ROC (across all splits) — DEHYDRATION ──
if len(all_dehy_cv_tprs) > 0:
    cv_tprs_d = np.array(all_dehy_cv_tprs)
    mean_tpr_cv_d = cv_tprs_d.mean(axis=0)
    mean_tpr_cv_d[-1] = 1.0
    mean_auc_cv_d = auc(mean_fpr_grid, mean_tpr_cv_d)
    std_tpr_cv_d = cv_tprs_d.std(axis=0)

    plt.figure(figsize=(8, 6))
    for i, tpr_i in enumerate(cv_tprs_d):
        plt.plot(mean_fpr_grid, tpr_i, alpha=0.15, linewidth=0.8)
    cv_aucs_arr_d = np.array(all_dehy_cv_aucs)
    plt.plot(mean_fpr_grid, mean_tpr_cv_d, color='darkorange', linewidth=2,
             label=f'Mean CV ROC (AUC = {cv_aucs_arr_d.mean():.3f} ± {cv_aucs_arr_d.std():.3f})')
    cv_upper_d = np.minimum(mean_tpr_cv_d + std_tpr_cv_d, 1)
    cv_lower_d = np.maximum(mean_tpr_cv_d - std_tpr_cv_d, 0)
    plt.fill_between(mean_fpr_grid, cv_lower_d, cv_upper_d, alpha=0.2, color='darkorange', label='± 1 SD')
    plt.plot([0, 1], [0, 1], 'k--', label='Chance')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Dehydration — Averaged CV ROC ({len(cv_tprs_d)} splits)')
    plt.legend(loc='lower right')
    os.makedirs(DEHY_OUTPUT_DIR, exist_ok=True)
    plt.savefig(os.path.join(DEHY_OUTPUT_DIR, 'ROC_Curve_dehydration_CV_averaged.tif'),
                format='tiff', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Mean CV AUC: {cv_aucs_arr_d.mean():.4f} ± {cv_aucs_arr_d.std():.4f}")

    dehy_y_pred_proba = dehy_model.predict_proba(X_test)[:, 1]  # keep for PR cell
else:
    print("No CV ROC data collected.")
    dehy_y_pred_proba = dehy_model.predict_proba(X_test)[:, 1]

#### Shape

In [ ]:
# NOTE: SHAP uses the best-performing split model (SHAP values don't average well across different models)
print(f"SHAP analysis uses best-split model (Test AUC={best_dehy_auc:.4f})")
calc_shap(dehy_model, X_test, 'dehydration', DEHY_OUTPUT_DIR)

#### Precision Recall

In [ ]:
# NOTE: PR curve uses best-split model data (bootstrapped CI from single best split)
print(f"PR curve uses best-split model (Test AUC={best_dehy_auc:.4f})")
calc_precision_recall(y_test, dehy_y_pred_proba, 'dehydration', DEHY_OUTPUT_DIR)

In [ ]:
# ── Save Dehydration results for publication section ──
X_test_dehy = X_test.copy()
y_test_dehy = y_test.copy()
X_train_dehy = X_train.copy()
y_train_dehy = y_train.copy()
dehy_preds = predictions.copy()
dehy_weights = sample_weights_train.copy()
df_dehy_final = df.copy()
print("Dehydration results saved.")

---
## 4. Publication Figures & Tables

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Publication setup: fonts, sizing, output directory
# ══════════════════════════════════════════════════════════════════
import matplotlib as mpl

PUB_DIR = '../../../Publication_Figures_Exports'
os.makedirs(PUB_DIR, exist_ok=True)

# Journal-quality defaults
mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
})

print(f"Publication figures will be saved to: {PUB_DIR}/")

### Table 1 — Patient Demographics & Feature Overview

In [ ]:
# ── Table 1a: Patient Demographics ──
# Reload personal data & export for demographics
_pers = pd.read_csv('personal_data.csv')
_exp = pd.read_csv('export.csv')

# Get demographics from export.csv (has study_id, age, gender, height, weight)
_demo = _exp.groupby('study_id').first().reset_index()

# Get unique patients from our dataset
patient_ids = sorted(all_data['ID'].unique())
n_patients = len(patient_ids)

# Demographics per patient
demo_rows = []
for pid in patient_ids:
    p = _demo[_demo['study_id'] == pid].iloc[0] if pid in _demo['study_id'].values else None
    tw_count = len(df_pain_final[df_pain_final['ID'] == pid]) if pid in df_pain_final['ID'].values else 0
    if p is not None:
        age_val = p.get('age', np.nan)
        gender_val = 'F' if p.get('gender', 0) == 0 else 'M'
        h = p.get('height', np.nan)
        w = p.get('weight', np.nan)
        bmi = w / ((h/100)**2) if pd.notna(h) and pd.notna(w) and h > 0 else np.nan
    else:
        age_val, gender_val, h, w, bmi = np.nan, '?', np.nan, np.nan, np.nan
    demo_rows.append({'ID': pid, 'Age': age_val, 'Gender': gender_val,
                      'Height (cm)': h, 'Weight (kg)': w, 'BMI': round(bmi, 1) if pd.notna(bmi) else np.nan,
                      'Time Windows': tw_count})

demo_df = pd.DataFrame(demo_rows)

# Summary statistics
print("=" * 70)
print("TABLE 1a: Patient Demographics")
print("=" * 70)
print(f"  N patients:   {n_patients}")
print(f"  Age:          {demo_df['Age'].mean():.1f} ± {demo_df['Age'].std():.1f} years "
      f"(range {demo_df['Age'].min():.0f}–{demo_df['Age'].max():.0f})")
n_f = (demo_df['Gender'] == 'F').sum()
n_m = (demo_df['Gender'] == 'M').sum()
print(f"  Gender:       {n_f} female ({100*n_f/n_patients:.0f}%), {n_m} male ({100*n_m/n_patients:.0f}%)")
print(f"  BMI:          {demo_df['BMI'].mean():.1f} ± {demo_df['BMI'].std():.1f} kg/m² "
      f"(range {demo_df['BMI'].min():.1f}–{demo_df['BMI'].max():.1f})")
print(f"  Time windows: {demo_df['Time Windows'].sum()} total, "
      f"{demo_df['Time Windows'].mean():.0f} ± {demo_df['Time Windows'].std():.0f} per patient")
print()
print(demo_df.to_string(index=False))
print()

# ── Table 1b: Feature Overview ──
# Get feature names from the pain model (after feature selection)
feature_names = list(X_train_pain.columns)

# Categorize features
categories = {
    'Vital Signs':       ['HeartRate', 'OxygenSaturation', 'HeartRateVariabilitySDNN', 'RestingHeartRate'],
    'Activity & Energy': ['ActiveEnergyBurned', 'BasalEnergyBurned', 'PhysicalEffort',
                          'AppleStandHour', 'AppleStandTime', 'AppleExerciseTime'],
    'Gait & Mobility':   ['DistanceWalkingRunning', 'StepCount', 'WalkingStepLength', 'WalkingSpeed',
                          'WalkingAsymmetryPercentage', 'WalkingDoubleSupportPercentage',
                          'WalkingHeartRateAverage', 'FlightsClimbed', 'StairAscentSpeed',
                          'StairDescentSpeed', 'SixMinuteWalkTestDistance'],
    'Demographics':      ['Age', 'Gender'],
    'Derived (diff)':    ['diff_'],
    'Derived (lag)':     ['_lag'],
    'Derived (rolling)': ['_sum_lag'],
}

print("=" * 70)
print("TABLE 1b: Feature Overview by Category")
print("=" * 70)
assigned = set()
for cat_name, prefixes in categories.items():
    matching = [f for f in feature_names
                if any(f.startswith(p) or p in f for p in prefixes)
                and f not in assigned]
    assigned.update(matching)
    print(f"  {cat_name}: {len(matching)} features")
    for f in sorted(matching):
        print(f"    - {f}")
unassigned = [f for f in feature_names if f not in assigned]
if unassigned:
    print(f"  Other: {len(unassigned)} features")
    for f in sorted(unassigned):
        print(f"    - {f}")
print(f"\nTotal features (Pain model): {len(feature_names)}")
print(f"Total features (Dehydration model): {len(X_train_dehy.columns)}")

### Figure 1 — Study Design Flowchart (CONSORT-style)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 10))
ax.set_xlim(0, 10)
ax.set_ylim(0, 14)
ax.axis('off')

box_kw = dict(boxstyle='round,pad=0.4', facecolor='#E8F0FE', edgecolor='#333333', linewidth=1.2)
box_kw2 = dict(boxstyle='round,pad=0.4', facecolor='#FFF3E0', edgecolor='#333333', linewidth=1.2)
box_kw3 = dict(boxstyle='round,pad=0.4', facecolor='#E8F5E9', edgecolor='#333333', linewidth=1.2)
txt_kw = dict(ha='center', va='center', fontsize=9, fontweight='normal')

# Row 1: Enrollment
ax.text(5, 13, 'Nursing home residents assessed\nfor eligibility (n=40 planned)',
        bbox=box_kw, **txt_kw)

# Arrow
ax.annotate('', xy=(5, 11.8), xytext=(5, 12.5),
            arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

# Row 2: Excluded
ax.text(8, 12.1, 'Excluded (n=24)\n• Declined (n=8)\n• Did not meet criteria (n=7)\n• Other reasons (n=9)',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFEBEE', edgecolor='#333', linewidth=1.2),
        ha='center', va='center', fontsize=8)
ax.annotate('', xy=(6.5, 12.1), xytext=(5.3, 12.1),
            arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

# Row 3: Enrolled
ax.text(5, 11.3, 'Enrolled and equipped with\nApple Watch Ultra 2 (n=16)',
        bbox=box_kw, **txt_kw)

# Arrow
ax.annotate('', xy=(5, 10.2), xytext=(5, 10.8),
            arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

# Row 4: Monitoring
ax.text(5, 9.7, '4-week continuous monitoring\n3× weekly clinical assessments\n(VAS pain, dehydration signs, vitals)',
        bbox=box_kw2, **txt_kw)

# Arrow
ax.annotate('', xy=(5, 8.4), xytext=(5, 9.1),
            arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

# Row 5: Data Processing
ax.text(5, 7.9, 'Data processing\n• 120-min time windows  • Feature aggregation\n• Gait imputation  • Lag & rolling features',
        bbox=box_kw2, ha='center', va='center', fontsize=8.5)

# Arrow splits
ax.annotate('', xy=(3, 6.5), xytext=(5, 7.3),
            arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))
ax.annotate('', xy=(7, 6.5), xytext=(5, 7.3),
            arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

# Row 6: Pain branch
n_pain = len(df_pain_final)
n_pain_pos = int(df_pain_final['pain_label'].sum())
ax.text(3, 6, f'Pain Estimation\nn={n_pain} time windows\n{n_pain_pos} positive ({100*n_pain_pos/n_pain:.0f}%)',
        bbox=box_kw3, **txt_kw)

# Row 6: Dehydration branch
n_dehy = len(df_dehy_final)
n_dehy_pos = int(df_dehy_final['dehydration_label'].sum())
ax.text(7, 6, f'Dehydration Estimation\nn={n_dehy} time windows\n{n_dehy_pos} positive ({100*n_dehy_pos/n_dehy:.0f}%)',
        bbox=box_kw3, **txt_kw)

# Arrows down
ax.annotate('', xy=(3, 4.5), xytext=(3, 5.3),
            arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))
ax.annotate('', xy=(7, 4.5), xytext=(7, 5.3),
            arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

# Row 7: Train/Test
n_train_p = len(X_train_pain)
n_test_p = len(X_test_pain)
ax.text(3, 4, f'Train: {n_train_p} | Test: {n_test_p}\n(4 patients held out)',
        bbox=box_kw3, ha='center', va='center', fontsize=8.5)

n_train_d = len(X_train_dehy)
n_test_d = len(X_test_dehy)
ax.text(7, 4, f'Train: {n_train_d} | Test: {n_test_d}\n(4 patients held out)',
        bbox=box_kw3, ha='center', va='center', fontsize=8.5)

# Arrows to model
ax.annotate('', xy=(3, 2.8), xytext=(3, 3.4),
            arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))
ax.annotate('', xy=(7, 2.8), xytext=(7, 3.4),
            arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

# Row 8: Model
ax.text(3, 2.3, 'Random Forest + Multi-Split\n(label-changing test patients)',
        bbox=box_kw2, ha='center', va='center', fontsize=8.5)
ax.text(7, 2.3, 'Random Forest + Multi-Split\n(label-changing test patients)',
        bbox=box_kw2, ha='center', va='center', fontsize=8.5)

# Arrows to eval
ax.annotate('', xy=(3, 1.1), xytext=(3, 1.7),
            arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))
ax.annotate('', xy=(7, 1.1), xytext=(7, 1.7),
            arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

# Row 9: Evaluation
ax.text(3, 0.6, 'Evaluation:\nTest AUC, Accuracy, F1\nSHAP Interpretability',
        bbox=box_kw3, ha='center', va='center', fontsize=8.5)
ax.text(7, 0.6, 'Evaluation:\nTest AUC, Accuracy, F1\nSHAP Interpretability',
        bbox=box_kw3, ha='center', va='center', fontsize=8.5)

fig.suptitle('Figure 1: Study Design and Analysis Pipeline', fontsize=13, fontweight='bold', y=0.98)
plt.savefig(os.path.join(PUB_DIR, 'Fig1_CONSORT.tif'), format='tiff', dpi=300)
plt.savefig(os.path.join(PUB_DIR, 'Fig1_CONSORT.png'), dpi=300)
plt.show()
print("Figure 1 saved.")

### Figure 2 — Per-Patient Label Distribution

In [ ]:
# Figure 2: Per-patient label distribution
required_dataframes = ["df_pain_final", "df_dehy_final"]
missing_dataframes = [name for name in required_dataframes if name not in globals()]
if missing_dataframes:
    raise RuntimeError(
        "Run the pain and dehydration data-preparation cells before Figure 2. "
        f"Missing: {', '.join(missing_dataframes)}"
    )

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5), sharey=False)

colors = {
    "negative": "#0072B2",  # Okabe-Ito blue
    "positive": "#D55E00",  # Okabe-Ito vermillion
}

# Pain
ax = axes[0]
pain_counts = df_pain_final.groupby("ID")["pain_label"].value_counts().unstack(fill_value=0)
for label in (0.0, 1.0):
    if label not in pain_counts.columns:
        pain_counts[label] = 0
pain_counts = pain_counts[[0.0, 1.0]].sort_index()

x = np.arange(len(pain_counts))
w = 0.38
ax.bar(x - w / 2, pain_counts[0.0], w, label="No pain (0)", color=colors["negative"], edgecolor="white", linewidth=0.8)
ax.bar(x + w / 2, pain_counts[1.0], w, label="Pain (1)", color=colors["positive"], edgecolor="white", linewidth=0.8)
ax.set_xlabel("Patient ID", fontsize=14)
ax.set_ylabel("Number of time windows", fontsize=14)
ax.set_title("A. Pain label distribution", fontsize=16, fontweight="bold", pad=12)
ax.set_xticks(x)
ax.set_xticklabels([f"ID{i}" for i in pain_counts.index], rotation=45, ha="right", fontsize=11)
ax.tick_params(axis="y", labelsize=12)
ax.legend(loc="upper right", framealpha=0.95, fontsize=11)
ax.grid(axis="y", alpha=0.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for i, patient_id in enumerate(pain_counts.index):
    if pain_counts.loc[patient_id, 0.0] == 0 or pain_counts.loc[patient_id, 1.0] == 0:
        ax.text(i, pain_counts.loc[patient_id].max() + 2, "*", ha="center", fontsize=18, color="#CC79A7", fontweight="bold")

# Dehydration
ax = axes[1]
dehy_counts = df_dehy_final.groupby("ID")["dehydration_label"].value_counts().unstack(fill_value=0)
for label in (0.0, 1.0):
    if label not in dehy_counts.columns:
        dehy_counts[label] = 0
dehy_counts = dehy_counts[[0.0, 1.0]].sort_index()

x = np.arange(len(dehy_counts))
ax.bar(x - w / 2, dehy_counts[0.0], w, label="Not dehydrated (0)", color=colors["negative"], edgecolor="white", linewidth=0.8)
ax.bar(x + w / 2, dehy_counts[1.0], w, label="Dehydrated (1)", color=colors["positive"], edgecolor="white", linewidth=0.8)
ax.set_xlabel("Patient ID", fontsize=14)
ax.set_ylabel("Number of time windows", fontsize=14)
ax.set_title("B. Dehydration label distribution", fontsize=16, fontweight="bold", pad=12)
ax.set_xticks(x)
ax.set_xticklabels([f"ID{i}" for i in dehy_counts.index], rotation=45, ha="right", fontsize=11)
ax.tick_params(axis="y", labelsize=12)
ax.legend(loc="upper right", framealpha=0.95, fontsize=11)
ax.grid(axis="y", alpha=0.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

for i, patient_id in enumerate(dehy_counts.index):
    if dehy_counts.loc[patient_id, 0.0] == 0 or dehy_counts.loc[patient_id, 1.0] == 0:
        ax.text(i, dehy_counts.loc[patient_id].max() + 2, "*", ha="center", fontsize=18, color="#CC79A7", fontweight="bold")

fig.suptitle("Figure 2: Per-patient label distribution (* = single-class patient)",
             fontsize=18, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PUB_DIR, "Fig2_Label_Distribution.tif"), format="tiff", dpi=300, bbox_inches="tight")
plt.savefig(os.path.join(PUB_DIR, "Fig2_Label_Distribution.png"), dpi=300, bbox_inches="tight")
plt.show()
print("Figure 2 saved.")

### Figure 3 — Pain ROC Curve (Averaged Held-Out LOPOCV)

In [ ]:
from sklearn.metrics import roc_curve, auc as sklearn_auc
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

def save_avg_holdout_roc(mean_fpr, tpr_curves, label_name, output_dir, fig_name, holdout_label, line_color):
    """Publication-quality averaged held-out ROC curve."""
    if len(tpr_curves) == 0:
        raise ValueError(f"No held-out ROC curves available for {label_name}.")

    tprs = np.asarray(tpr_curves)
    mean_tpr = tprs.mean(axis=0)
    mean_tpr[-1] = 1.0
    auc_values = np.array([sklearn_auc(mean_fpr, tpr_i) for tpr_i in tprs])
    mean_auc = sklearn_auc(mean_fpr, mean_tpr)
    std_auc = auc_values.std()
    std_tpr = tprs.std(axis=0)

    fig, ax = plt.subplots(figsize=(6, 6))
    for tpr_i in tprs:
        ax.plot(mean_fpr, tpr_i, alpha=0.12, linewidth=0.8, color=line_color)

    ax.plot(mean_fpr, mean_tpr, color=line_color, linewidth=2.2,
            label=f'Mean held-out ROC (AUC = {mean_auc:.2f} ± {std_auc:.2f})')
    ax.fill_between(
        mean_fpr,
        np.maximum(mean_tpr - std_tpr, 0),
        np.minimum(mean_tpr + std_tpr, 1),
        alpha=0.18, color=line_color, label='± 1 SD'
    )
    ax.plot([0, 1], [0, 1], 'k--', lw=0.8, alpha=0.5, label='Chance')

    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_xlabel('False Positive Rate (1 - Specificity)')
    ax.set_ylabel('True Positive Rate (Sensitivity)')
    ax.set_title(f'ROC Curve - {label_name.capitalize()} Estimation')
    ax.text(
        0.04, 0.06, holdout_label, transform=ax.transAxes, fontsize=9,
        bbox=dict(boxstyle='round,pad=0.25', facecolor='white', edgecolor='none', alpha=0.9)
    )
    ax.legend(loc='lower right', framealpha=0.95)
    ax.set_aspect('equal')
    plt.tight_layout()

    plt.savefig(os.path.join(output_dir, fig_name + '.tif'), format='tiff', dpi=300)
    plt.savefig(os.path.join(output_dir, fig_name + '.png'), dpi=300)
    plt.close(fig)
    print(f"Saved {fig_name}")
    print(f"Mean held-out ROC AUC: {mean_auc:.4f} ± {std_auc:.4f}")
    return mean_auc, std_auc

def save_best_split_roc(model, X_train, y_train, X_test, y_test, sample_weights,
                        label_name, output_dir, fig_name, n_bootstraps=200):
    """Supplementary best-split ROC with CV curve and test curve + bootstrap CI."""

    kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    cv_proba = np.zeros(len(y_train))
    for train_idx, val_idx in kf.split(X_train, y_train):
        m = clone(model)
        w = sample_weights.iloc[train_idx] if sample_weights is not None else None
        m.fit(X_train.iloc[train_idx], y_train.iloc[train_idx],
              classifier__sample_weight=w)
        cv_proba[val_idx] = m.predict_proba(X_train.iloc[val_idx])[:, 1]
    fpr_cv, tpr_cv, _ = roc_curve(y_train, cv_proba)
    auc_cv = sklearn_auc(fpr_cv, tpr_cv)

    y_pred_proba = model.predict_proba(X_test)[:, 1]
    fpr_test, tpr_test, _ = roc_curve(y_test, y_pred_proba)
    auc_test = sklearn_auc(fpr_test, tpr_test)

    rng = np.random.RandomState(42)
    mean_fpr = np.linspace(0, 1, 200)
    boot_tprs = []
    y_test_np = np.asarray(y_test)
    y_proba_np = np.asarray(y_pred_proba)

    for _ in range(n_bootstraps):
        idx = rng.randint(0, len(y_test_np), len(y_test_np))
        if np.unique(y_test_np[idx]).size < 2:
            continue
        fpr_b, tpr_b, _ = roc_curve(y_test_np[idx], y_proba_np[idx])
        boot_tprs.append(np.interp(mean_fpr, fpr_b, tpr_b))

    boot_tprs = np.array(boot_tprs)
    tpr_lo = np.percentile(boot_tprs, 5, axis=0)
    tpr_hi = np.percentile(boot_tprs, 95, axis=0)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], 'k--', lw=0.8, alpha=0.5, label='Chance')
    ax.plot(fpr_cv, tpr_cv, color='#1976D2', lw=2,
            label=f'3-fold CV (AUC = {auc_cv:.2f})')
    ax.plot(fpr_test, tpr_test, color='#D32F2F', lw=2,
            label=f'Held-out test (AUC = {auc_test:.2f})')
    ax.fill_between(mean_fpr, tpr_lo, tpr_hi, color='#D32F2F', alpha=0.12,
                    label='Test 90% CI (bootstrap)')

    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_xlabel('False Positive Rate (1 - Specificity)')
    ax.set_ylabel('True Positive Rate (Sensitivity)')
    ax.set_title(f'ROC Curve - {label_name.capitalize()} Estimation')
    ax.legend(loc='lower right', framealpha=0.95)
    ax.set_aspect('equal')
    plt.tight_layout()

    plt.savefig(os.path.join(output_dir, fig_name + '.tif'), format='tiff', dpi=300)
    plt.savefig(os.path.join(output_dir, fig_name + '.png'), dpi=300)
    plt.close(fig)
    print(f"Saved {fig_name}")
    return auc_cv, auc_test

pain_auc_holdout_mean, pain_auc_holdout_sd = save_avg_holdout_roc(
    mean_fpr_grid, all_pain_test_tprs, 'pain', PUB_DIR, 'Fig3_ROC_Pain',
    'Averaged held-out ROC across 8 LOPO folds', '#1976D2'
 )

supp_pain_auc_cv, supp_pain_auc_test = save_best_split_roc(
    pain_model, X_train_pain, y_train_pain, X_test_pain, y_test_pain,
    pain_weights, 'pain', PUB_DIR, 'Supplement_FigS1_ROC_Pain_BestSplit'
 )

### Figure 4 — Dehydration ROC Curve (Averaged Held-Out Splits)

In [ ]:
dehy_auc_holdout_mean, dehy_auc_holdout_sd = save_avg_holdout_roc(
    mean_fpr_grid, all_dehy_test_tprs, 'dehydration', PUB_DIR, 'Fig4_ROC_Dehydration',
    'Averaged held-out ROC across 15 patient-level splits', '#D32F2F'
 )

supp_dehy_auc_cv, supp_dehy_auc_test = save_best_split_roc(
    dehy_model, X_train_dehy, y_train_dehy, X_test_dehy, y_test_dehy,
    dehy_weights, 'dehydration', PUB_DIR, 'Supplement_FigS2_ROC_Dehydration_BestSplit'
 )

### Figure 5 — SHAP Beeswarm (Pain)

In [ ]:
FEATURE_NAME_MAP = {
    "ActiveEnergyBurned": "Active energy",
    "BasalEnergyBurned": "Basal energy",
    "DistanceWalkingRunning": "Walking/running distance",
    "HeartRate": "HR",
    "HeartRateVariabilitySDNN": "HRV-SDNN",
    "OxygenSaturation": "SpO2",
    "PhysicalEffort": "Physical effort",
    "RestingHeartRate": "Resting HR",
    "SixMinuteWalkTestDistance": "Six-minute walk distance",
    "StairAscentSpeed": "Stair ascent speed",
    "StairDescentSpeed": "Stair descent speed",
    "StepCount": "Step count",
    "WalkingAsymmetry": "Walking asymmetry",
    "WalkingDoubleSupportPercentage": "Walking double support",
    "WalkingHeartRateAverage": "Walking HR average",
    "WalkingSpeed": "Walking speed",
}

STAT_NAME_MAP = {
    "max": "maximum",
    "median": "median",
    "min": "minimum",
    "sum": "sum",
}

WINDOW_HOURS = 2


def format_feature_name(feature_name):
    """Convert raw model feature columns into publication-friendly labels."""
    name = feature_name
    prefix = ""
    time_label = ""

    if name.startswith("diff_"):
        prefix = "Change in "
        name = name[5:]

    for suffix in ("_lag1", "_lag3", "_lag6"):
        if name.endswith(suffix):
            hours_ago = int(suffix[-1]) * WINDOW_HOURS
            time_label = f" ({hours_ago} h ago)"
            name = name[: -len(suffix)]
            break

    if name.endswith("_sum_sum_lag"):
        name = name[: -len("_sum_sum_lag")]
        time_label = " (rolling sum)"

    parts = name.split("_")
    base = FEATURE_NAME_MAP.get(parts[0], parts[0])
    stats = [STAT_NAME_MAP.get(part, part) for part in parts[1:] if part]
    stat_label = f" ({', '.join(stats)})" if stats else ""
    return f"{prefix}{base}{stat_label}{time_label}"


def pub_shap(model, X_test, label_name, output_dir, fig_name, max_display=20):
    """Publication-quality SHAP beeswarm plot."""
    explainer = shap.Explainer(model.predict, X_test)
    shap_values = explainer(X_test)
    X_display = X_test.copy()
    X_display.columns = [format_feature_name(col) for col in X_display.columns]

    fig = plt.figure(figsize=(10.5, 9.2))
    shap.summary_plot(shap_values.values, X_display, show=False, max_display=max_display,
                      plot_size=None)
    ax = plt.gca()
    ax.set_title(f"SHAP Feature Importance - {label_name.capitalize()} Estimation", fontsize=15, pad=12)
    ax.set_xlabel("SHAP value (impact on model output)", fontsize=13)
    ax.tick_params(axis="x", labelsize=11)
    ax.tick_params(axis="y", labelsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, fig_name + ".tif"), format="tiff", dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(output_dir, fig_name + ".png"), dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved {fig_name}")

pub_shap(pain_model, X_test_pain, "pain", PUB_DIR, "Fig5_SHAP_Pain")

### Figure 6 — SHAP Beeswarm (Dehydration)

In [ ]:
pub_shap(dehy_model, X_test_dehy, 'dehydration', PUB_DIR, 'Fig6_SHAP_Dehydration')

---
### Supplementary — Classification Reports

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, brier_score_loss,
                              precision_score, recall_score, f1_score)

def full_report(model, X_train, y_train, X_test, y_test, label_name, sample_weights=None):
    """Print comprehensive classification report for train and test."""
    print("=" * 70)
    print(f"  {label_name.upper()} - Full Classification Report")
    print("=" * 70)

    # -- TRAINING --
    y_pred_train = model.predict(X_train)
    y_proba_train = model.predict_proba(X_train)[:, 1]
    acc_train = accuracy_score(y_train, y_pred_train)
    auc_train = roc_auc_score(y_train, y_proba_train)

    print(f"\n-- Training Set (n={len(y_train)}) --")
    print(f"  Accuracy:  {acc_train:.4f}")
    print(f"  AUC:       {auc_train:.4f}")
    print(classification_report(y_train, y_pred_train, digits=3,
          target_names=['Negative', 'Positive']))

    # -- TEST --
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc_val = roc_auc_score(y_test, y_proba)
    brier = brier_score_loss(y_test, y_proba)

    print(f"-- Test Set (n={len(y_test)}) --")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    print(f"  AUC:       {auc_val:.4f}")
    print(f"  Brier:     {brier:.4f}")
    print()
    print(classification_report(y_test, y_pred, digits=3,
          target_names=['Negative', 'Positive']))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    print("  Confusion Matrix (test):")
    print(f"    TN={cm[0,0]}  FP={cm[0,1]}")
    print(f"    FN={cm[1,0]}  TP={cm[1,1]}")
    print()

    # -- Model hyperparameters --
    clf = model.named_steps.get('classifier', model[-1])
    print(f"  Best hyperparameters:")
    for p in ['n_estimators', 'max_depth', 'min_samples_split', 'min_samples_leaf',
              'max_features', 'class_weight']:
        print(f"    {p}: {getattr(clf, p, 'N/A')}")
    print("=" * 70)

    return {'Accuracy': acc, 'Precision': prec, 'Recall': rec,
            'F1': f1, 'AUC': auc_val, 'Brier': brier}

print()
pain_metrics = full_report(pain_model, X_train_pain, y_train_pain,
                           X_test_pain, y_test_pain, 'Pain', pain_weights)
print()
dehy_metrics = full_report(dehy_model, X_train_dehy, y_train_dehy,
                           X_test_dehy, y_test_dehy, 'Dehydration', dehy_weights)

### Summary Metrics Table (for direct copy into manuscript)

In [ ]:
# Create a clean best-split metrics comparison table for the publication appendix
metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC (Test)', 'Brier Score'],
    'Pain': [pain_metrics['Accuracy'], pain_metrics['Precision'],
             pain_metrics['Recall'], pain_metrics['F1'],
             pain_metrics['AUC'], pain_metrics['Brier']],
    'Dehydration': [dehy_metrics['Accuracy'], dehy_metrics['Precision'],
                    dehy_metrics['Recall'], dehy_metrics['F1'],
                    dehy_metrics['AUC'], dehy_metrics['Brier']],
})
metrics_df['Pain'] = metrics_df['Pain'].map('{:.3f}'.format)
metrics_df['Dehydration'] = metrics_df['Dehydration'].map('{:.3f}'.format)

print("\nTable — Best-Split Model Performance Comparison")
print(metrics_df.to_string(index=False))

# Save as CSV for inspection; pooled manuscript metrics are stored in the task result folders
metrics_df.to_csv(os.path.join(PUB_DIR, 'metrics_best_split_comparison.csv'), index=False)
print(f"\nSaved to {PUB_DIR}/metrics_best_split_comparison.csv")

In [ ]:
print("\n" + "=" * 70)
print("Publication files generated:")
print("=" * 70)
for f in sorted(os.listdir(PUB_DIR)):
    fpath = os.path.join(PUB_DIR, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {f:45s} {size_kb:7.1f} KB")
print("=" * 70)